# IV-9 研究テーマを決める：研究地図・論文の読み方・小さな再現・必要な試行の数（ノートブック版）

教科書の IV-9（id = p4c9）の 6 節のプログラムを、手順 A〜E の順に 1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac（Apple シリコン）で全部で約 4〜5 分（3 節の再現を試しの設定にしたとき）。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 見本の題目 15 件から研究地図の数を出し、分類の規則を変えてみる | A | `thesis_map.py` |
| 2 | 論文の書誌を DOI から確かめる（**インターネットが必要**） | B | `crossref_check.py`・`my_refs.csv` |
| 3 | 論文の図を 1 枚、小さく再現する（窓長と分離性能。**試しの設定**）と、耳で確かめる | C | `repro_window.py` |
| 4 | 改良の比較に要る試行の数を出す | D | `seeds_needed.py` |
| 5 | テーマ候補 3 つのワークシートを作り、点検する | E | `theme_sheet.py` |
| 6 | 確認問題の答え合わせ | 7 節 | `quiz_check_iv9.py` |

- 次のセルたちが、共通ライブラリ（IV-2 の `common/` の `__init__.py`・`stft.py`・`audio_io.py`・`plotting.py`）と、第7章の `bss.py`、IV-6 の `bss6.py`、この章の `seeds_needed.py`・`theme_sheet.py`（6 節の確認問題が読み込む）を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の各章の 6 節と同じなので、ほかのファイルは要りません。
- 各節のコードは 6 節のプログラムそのものです。変えたのは、ファイルの場所（`__file__` の代わりにノートブックのフォルダを使う。`sys.path.insert` も同じ）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、日本語のフォントの候補を足すこと、コマンドの引数を `sys.argv` で渡すこと（同じプログラムを引数を変えて何度も実行するところは、`main()` をくり返し呼ぶか、書き出したファイルを `%run` で実行する）、出力をいったん `cap` に受け取って次のセルで表示し、`ok()` で教科書の手順の「確かめること」と 7 節の結果の例を確かめることだけです。1 節の「規則を自分で直してみる」セルはノートブックで足したものです。
- **3 節（手順 C）の再現は重い**ので、ノートブックでは試しの設定（`--seeds 4`：種 4 通り × 窓長 6 通り。執筆時の Mac で約 4 分）を既定にしています。教科書の本番（種 10 通り、40〜60 分）にするときは、セルのコメントのとおりに `--seeds 4` を消してください。種 0〜3 は本番の 10 通りの一部なので値は 7 節の表に近くなりますが、平均・区間・「ILRMA が上の種」の数は合いません。`ok()` は、種の数に関係なく成り立つこと（oracleW は窓が長いほど上がる、256 ms で ILRMA が AuxIVA より上、1024 ms では両方とも改善量がほぼ 0 以下、など）と、7 節の平均からの大きなずれがないことだけを確かめます。
- **入れていないもの**（コードがない、または実際の行動が要る手順）：`setup_iv9.sh`（ファイルを `~/home-lab/` に置く準備。ここでは `%%writefile` が代わり）と `run_iv9.sh`（全部を順に実行するシェルスクリプト。このノートブックが代わり）、手順 A の研究室のページ・KAKEN・業績一覧を読むこと、手順 B の論文を 3 段階で読むこと（ワークシートは紙かノートに書く）、手順 D の事前登録を Git にコミットすること、手順 E の問いの種の書き出しと採点、手順 F の研究室訪問の準備。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。保存した wav（最大 −10 dBFS）をそのままの大きさで鳴らすため、`IPython.display.Audio(..., normalize=False)` を使います。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を一番小さくしてください。
- 話し声は librosa の例の音源 `libri1`・`libri3` を使います（初回はダウンロードします。ネットワークが要ります）。
- 研究室の題目の一覧（1 節）は、説明のための架空の見本 15 件です。配属先で使うときは、研究室のページの題目の一覧を開いて `TITLES` を書き換えてください。
- 数値は乱数の種を変えなければ教科書とほぼ同じですが、pyroomacoustics・librosa・scipy の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。
- Colab では、最初の準備のセルで pyroomacoustics（と fast_bss_eval）を pip で入れます。pyroomacoustics はその場でビルドすることがあり、**数分かかることがあります**（止まって見えても、終わるまで待ってください）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"), ("statsmodels", "statsmodels"),
                 ("pyroomacoustics", "pyroomacoustics"), ("fast_bss_eval", "fast_bss_eval")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, statsmodels, pyroomacoustics, fast_bss_eval, librosa
print("OK", numpy.__version__, scipy.__version__, statsmodels.__version__, pyroomacoustics.__version__, librosa.__version__)
print("（教科書は numpy 2.5.5・scipy 1.18.1・statsmodels 0.15.0・pyroomacoustics 0.10.1・fast_bss_eval 0.1.4・librosa 1.0.1 で確かめた）")
import os
for d in ["common", "out"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の 8 つのセルで、ほかの章のファイル（IV-2 の `common/`、第7章の `bss.py`、IV-6 の `bss6.py`）と、この章の `seeds_needed.py`・`theme_sheet.py` を書き出します。

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/stft.py
"""第IV部の共通ライブラリ：短時間フーリエ変換（STFT）と逆 STFT（IV-2 で定義）

約束（第IV部のすべての章で同じ）
  X = stft(x, L=1024, S=256, window="hann")
      x : 形 (n,) か (n, M)（M はマイクの数）
      X : 形 (F, T) か (F, T, M)、F = L//2 + 1、T = 1 + n//S（複素数、complex128）
  x = istft(X, L, S, window="hann", length=None)
      stft で作った X なら、元の x に戻る（完全再構成。誤差は 1e-12 程度）

決め方（librosa.stft の既定と同じ。第7章の bss.py・nmf.py の STFT と数値が一致する）
  ・窓は「周期的な」窓（scipy.signal.get_window の既定。hann なら w[n] = sin²(πn/L)）
  ・フレーム t の窓の中心が、信号の時刻 n = t·S に来る（前後に L//2 個のゼロを足す）
  ・位相の基準は窓の始まり（phase="start"）。phase="center" にすると窓の中心が基準になり、
    scipy.signal.ShortTimeFFT の既定（phase_shift=0）と同じになる
  ・正規化はしない（窓を掛けて FFT した値そのもの）
  ・逆 STFT の合成窓は、最小二乗の意味で最適な窓 g[n] = w[n] / Σ_t w²[n − tS]
    （Griffin & Lim 1984。端で重なりが少ないところも、実際に重なっているフレームだけで割る）
"""
import numpy as np
from scipy.signal import get_window as _scipy_get_window

__all__ = ["get_window", "stft", "istft", "window_sum", "dual_window",
           "bin_freqs", "frame_times", "n_frames"]


def get_window(window, L):
    """窓を長さ L の配列で返す。window は名前（"hann"、"hamming"、("kaiser", 8) など）か配列。
    名前のときは周期的な窓（fftbins=True）。"sqrthann" は hann の平方根（分析と合成に同じ窓を使うとき）。"""
    if isinstance(window, str) and window == "sqrthann":
        return np.sqrt(_scipy_get_window("hann", L, fftbins=True))
    if isinstance(window, (str, tuple)):
        return _scipy_get_window(window, L, fftbins=True).astype(float)
    w = np.asarray(window, dtype=float)
    if w.shape != (L,):
        raise ValueError(f"窓の長さ {w.shape} が L = {L} と合いません")
    return w


def n_frames(n, S):
    """長さ n の信号のフレームの数 T = 1 + n//S"""
    return 1 + n // S


def bin_freqs(L, fs):
    """周波数のビン k = 0, …, L//2 の周波数 [Hz]：k·fs/L"""
    return np.arange(L // 2 + 1) * fs / L


def frame_times(T, S, fs):
    """フレーム t = 0, …, T−1 の窓の中心の時刻 [s]：t·S/fs"""
    return np.arange(T) * S / fs


def _phase_factor(L, F, phase):
    """位相の基準を「窓の中心」にするときに掛ける因子 exp(+j2πk(L//2)/L)（L が偶数なら (−1)^k）"""
    if phase == "start":
        return None
    if phase == "center":
        k = np.arange(F)
        if L % 2 == 0:
            return (-1.0) ** k + 0j                      # L が偶数なら exp(jπk) = (−1)^k を正確に
        return np.exp(2j * np.pi * k * (L // 2) / L)
    raise ValueError('phase は "start" か "center"')


def stft(x, L=1024, S=256, window="hann", phase="start"):
    """短時間フーリエ変換。X[k, t] = Σ_{m=0}^{L−1} w[m] x[tS + m − L//2] e^{−j2πkm/L}（範囲外の x は 0）"""
    x = np.asarray(x, dtype=float)
    if x.ndim not in (1, 2):
        raise ValueError("x の形は (n,) か (n, M)")
    if not 0 < S <= L:
        raise ValueError("シフト S は 1 以上 L 以下")
    w = get_window(window, L)
    n = x.shape[0]
    T = n_frames(n, S)
    pad = L // 2
    # 前に L//2、後ろはフレームが全部入るだけのゼロを足す
    total = max((T - 1) * S + L, pad + n)
    xp = np.zeros((total,) + x.shape[1:])
    xp[pad:pad + n] = x
    idx = np.arange(L)[None, :] + S * np.arange(T)[:, None]          # (T, L) 各フレームの番号
    frames = xp[idx] * (w[None, :] if x.ndim == 1 else w[None, :, None])   # (T, L) か (T, L, M)
    X = np.fft.rfft(frames, axis=1)                                   # (T, F) か (T, F, M)
    X = np.moveaxis(X, 0, 1)                                          # (F, T) か (F, T, M)
    c = _phase_factor(L, X.shape[0], phase)
    if c is not None:
        X = X * (c[:, None] if X.ndim == 2 else c[:, None, None])
    return X


def window_sum(w, S, T):
    """T 個のフレームを S ずつずらして重ねたときの Σ_t w²[n − tS]（長さ (T−1)S + L）"""
    L = len(w)
    out = np.zeros((T - 1) * S + L)
    for t in range(T):
        out[t * S:t * S + L] += w ** 2
    return out


def dual_window(w, S):
    """無限に続くフレームを考えたときの最適な合成窓（双対窓）g[n] = w[n] / Σ_k w²[n + kS]
    （scipy.signal.ShortTimeFFT の dual_win と同じもの）"""
    L = len(w)
    acc = np.zeros(L)
    for k in range(-(L // S) - 1, L // S + 2):
        lo, hi = max(0, k * S), min(L, L + k * S)
        if lo < hi:
            acc[lo:hi] += w[lo - k * S:hi - k * S] ** 2
    if np.any(acc <= 0):
        raise ValueError("NOLA の条件が成り立たない（Σ w² が 0 の点がある）")
    return w / acc


def istft(X, L, S, window="hann", length=None, phase="start"):
    """逆 STFT（重み付き重ね合わせ加算）。
    y[n] = Σ_t w[n − tS]·ifft(X[:, t])[n − tS] / Σ_t w²[n − tS]
    X が本物の STFT なら完全再構成、加工した X なら「STFT が X に最も近い信号」（最小二乗）を返す。"""
    X = np.asarray(X)
    if X.ndim not in (2, 3):
        raise ValueError("X の形は (F, T) か (F, T, M)")
    F, T = X.shape[:2]
    if F != L // 2 + 1:
        raise ValueError(f"X の周波数の数 {F} が L//2+1 = {L // 2 + 1} と合いません")
    w = get_window(window, L)
    c = _phase_factor(L, F, phase)
    if c is not None:
        X = X * (np.conj(c)[:, None] if X.ndim == 2 else np.conj(c)[:, None, None])
    frames = np.fft.irfft(X, n=L, axis=0)                            # (L, T) か (L, T, M)
    total = (T - 1) * S + L
    y = np.zeros((total,) + X.shape[2:])
    wb = w if X.ndim == 2 else w[:, None]
    for t in range(T):
        y[t * S:t * S + L] += wb * frames[:, t]
    wss = window_sum(w, S, T)
    pad = L // 2
    if length is None:
        length = (T - 1) * S
    end = min(pad + length, total)
    seg = wss[pad:end]
    if np.any(seg < 1e-10 * wss.max()):
        raise ValueError("NOLA の条件が成り立たない：窓の2乗の和が 0 の時刻があり、そこは戻せない")
    out = np.zeros((length,) + X.shape[2:])
    out[:end - pad] = y[pad:end] / (seg if X.ndim == 2 else seg[:, None])
    return out

In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x

In [ ]:
%%writefile common/plotting.py
"""第IV部の共通ライブラリ：図の設定とスペクトログラムの表示（IV-2 で定義）

  import matplotlib; matplotlib.use("Agg")      # 図をファイルに保存するだけのとき（第7章と同じ）
  from common.plotting import specshow
  specshow(X, fs, S, ax=ax, title="…")          # X は stft の結果（複素数）か、振幅・パワー

日本語のフォントは既存の章と同じ（Hiragino Sans）。
"""
import numpy as np
import matplotlib

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

__all__ = ["specshow", "to_db"]


def to_db(X, power=False, ref=None, floor_db=-120.0):
    """X（複素数・振幅・パワー）を dB にする。振幅は 20log10、パワーは 10log10。
    ref を与えないと、最大値を 0 dB にする"""
    A = np.abs(X)
    if power:
        D = 10 * np.log10(A + 1e-20)
    else:
        D = 20 * np.log10(A + 1e-20)
    D -= (D.max() if ref is None else (10 if power else 20) * np.log10(ref))
    return np.maximum(D, floor_db)


def specshow(X, fs, S, ax=None, power=False, db_range=80.0, fmax=None, title=None, cmap="magma",
             colorbar=True):
    """スペクトログラムを表示する。X は (F, T)。横軸はフレームの窓の中心の時刻 t·S/fs [s]、
    縦軸はビンの周波数 k·fs/L [Hz]（L = 2(F−1)）。色は最大値から db_range [dB] 下までを表示する。"""
    import matplotlib.pyplot as plt
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("specshow に渡す X は (F, T)。多チャネルなら X[:, :, m] のように1つ選ぶ")
    F, T = X.shape
    L = 2 * (F - 1)
    D = to_db(X, power=power)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    df, dt = fs / L, S / fs
    # 各ビン・各フレームの値を、その中心の周りの長方形で塗る
    extent = (-dt / 2, (T - 1) * dt + dt / 2, -df / 2, (F - 1) * df + df / 2)
    im = ax.imshow(D, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-db_range, vmax=0, interpolation="nearest")
    ax.set_xlabel("時間 [s]")
    ax.set_ylabel("周波数 [Hz]")
    if fmax is not None:
        ax.set_ylim(0, fmax)
    if title:
        ax.set_title(title)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="dB（最大を 0）")
    return im

In [ ]:
%%writefile bss.py
# 第7章の ~/home-lab/ch7/bss.py（AuxIVA・ILRMA の実装）と同じもの
#!/usr/bin/env python3
"""第7章：AuxIVA と ILRMA を NumPy でゼロから実装する（決定系：音源の数 N ＝ マイクの数 M）

記号（論文の周波数・時間の番号 i, j を、この章では f, t と書く。T はフレームの数で、
      論文の基底行列 T_n（ここでは Tb）とは別のもの）
  X : 観測の STFT。形は (F, T, M)。x_ft = [x_ft1, …, x_ftM]ᵀ
  W : 分離行列。形は (F, N, M)。y_ft = W_f x_ft。W_f の n 行目が w_nfᴴ
  Y : 分離信号 (F, T, N)、P = |Y|²
  r : 音源 n の分散（音源モデル）。AuxIVA は r_nt = √(Σ_f |y_nft|²)（周波数をまとめた大きさ）、
      ILRMA は r_nft = (T_n V_n)_ft（音源ごとの NMF）
更新（反復射影法 IP）
  U_nf = (1/T) Σ_t x_ft x_ftᴴ / r_n(ft)     （AuxIVA では φ(r)=1/r を掛ける）
  w_nf ← (W_f U_nf)⁻¹ e_n、 w_nf ← w_nf / √(w_nfᴴ U_nf w_nf)
コスト（負の対数尤度。反復で増えないことを確かめる）
  ILRMA : Σ_n Σ_ft (|y_nft|²/r_nft + log r_nft) − 2T Σ_f log|det W_f|
  AuxIVA: Σ_n Σ_t r_nt − T Σ_f log|det W_f|      （ラプラス分布の音源モデル）
          log|det| の係数が ILRMA の半分なのは、尺度を変えたラプラス分布の負の対数尤度を2で割ったものだから。
          定数倍なので、反復で増えないことの確認には影響しない
"""
import numpy as np

EPS = 1e-10


def demix(W, X):
    """Y = W X（周波数ごと）"""
    return np.einsum("fnm,ftm->ftn", W, X)


def ip_update(W, X, phi, n):
    """反復射影法で W の n 行目を更新する。phi は (F, T) の重み（1/r）"""
    F, T, M = X.shape
    U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), phi) / T          # (F, M, M)
    WU = W @ U
    e = np.zeros((F, M, 1), dtype=complex)
    e[:, n, 0] = 1.0
    w = np.linalg.solve(WU, e)[:, :, 0]                               # (F, M)
    denom = np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))
    w = w / (denom[:, None] + EPS)
    W[:, n, :] = w.conj()
    return W


def logdet(W):
    return np.sum(np.log(np.abs(np.linalg.det(W)) + EPS))


def projection_back(Y, W, ref=0):
    """スケールのあいまいさを、基準マイク ref で観測した大きさに合わせて直す"""
    A = np.linalg.inv(W)                                              # (F, M, N) 混合行列の推定
    return Y * A[:, ref, :][:, None, :]


def auxiva(X, n_iter=100, return_cost=False):
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    for _ in range(n_iter):
        for n in range(M):
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y[:, :, n]) ** 2, axis=0)) + EPS   # (T,)
            W = ip_update(W, X, np.broadcast_to(1.0 / r, (F, T)), n)
        if return_cost:
            Y = demix(W, X)
            r = np.sqrt(np.sum(np.abs(Y) ** 2, axis=0)) + EPS           # (T, N)
            hist.append(float(np.sum(r) - T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)


def ilrma(X, n_basis=4, n_iter=100, seed=0, return_cost=False):
    F, T, M = X.shape
    rng = np.random.default_rng(seed)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    Tb = rng.uniform(0.1, 1.0, (M, F, n_basis))                       # 基底 T_n
    Vb = rng.uniform(0.1, 1.0, (M, n_basis, T))                       # アクティベーション V_n
    R = Tb @ Vb                                                       # (N, F, T)
    hist = []
    for _ in range(n_iter):
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)               # (N, F, T)
        # 音源モデルの更新（IS ダイバージェンスの NMF。補助関数法の乗法更新）
        Tb *= np.sqrt(((P / R ** 2) @ np.swapaxes(Vb, 1, 2)) / ((1 / R) @ np.swapaxes(Vb, 1, 2) + EPS))
        R = Tb @ Vb + EPS
        Vb *= np.sqrt((np.swapaxes(Tb, 1, 2) @ (P / R ** 2)) / (np.swapaxes(Tb, 1, 2) @ (1 / R) + EPS))
        R = Tb @ Vb + EPS
        # 分離行列の更新（IP）
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
        # スケールの正規化（W の各行と、音源モデルの大きさをそろえる。結果は変わらない）
        Y = demix(W, X)
        lam = np.sqrt(np.mean(np.abs(Y) ** 2, axis=(0, 1))) + EPS     # (N,)
        W /= lam[None, :, None]
        R /= lam[:, None, None] ** 2
        Tb /= lam[:, None, None] ** 2
        if return_cost:
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            hist.append(float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W)))
    Y = projection_back(demix(W, X), W)
    return (Y, W, np.array(hist)) if return_cost else (Y, W)

In [ ]:
%%writefile bss6.py
# IV-6 の ~/home-lab/partIV/iv6/bss6.py と同じもの（ファイルの場所の 3 行だけ変えた）
#!/usr/bin/env python3
"""IV-6：第7章の bss.py をそのまま使い、理論の確かめに要る道具を足す（この章の共通部品）

第7章の ~/home-lab/ch7/bss.py を import する（demix・ip_update・logdet・projection_back・auxiva・ilrma）。
配列の形は第7章と同じ：X は (F, T, M)、W は (F, N, M)、y_ft = W_f x_ft、Y は (F, T, N)。

この章で足すもの
  sources(names, fs, sec)          : librosa の例の音源を読み、長さと大きさをそろえる
  simulate(src, fs, rt60, seed, …) : pyroomacoustics の鏡像法で M 本のマイクの観測を作る（rt60=0 は無響）
  fdica(X, n_iter)                 : 周波数ごとに独立な ICA（AuxICA）。パーミュテーションは解かない
  align_power_ratio(Y, W)          : パワー比の相関で順番をそろえる（ブラインド。Sawada らの方法の簡易版）
  align_oracle(Y, ref_img)         : 正解を使って順番をそろえる（上限を見るための oracle）
  swapped_stats(perms, energy_db)  : 順番が多数派と違うビンを、エネルギーのあるビンだけで数える
  ilrma_v(X, K, n_iter, …)         : 第7章の ilrma と同じ計算に、初期値・更新の順番・正規化を選ぶ引数を足したもの
  oracle_w(X, img)                 : 正解の像に最小二乗で合わせた分離行列（周波数ごとの線形分離の上限の目安）
  ip_oracle(X, img, n_iter)        : 音源モデルに正解のパワーを入れた IP（音源モデルが完璧なときの上限）
  cost_ilrma / cost_auxiva         : 負の対数尤度（第7章と同じ式）
  to_time / evaluate               : 逆 STFT と BSS Eval（fast_bss_eval）
"""
import pathlib
import sys

import numpy as np

sys.path.insert(0, ".")   # ノートブック版：common/ と第7章の bss.py は、このノートブックの隣に書き出してある

import bss                                              # noqa: E402  第7章の実装
from bss import demix, ip_update, logdet, projection_back, EPS   # noqa: E402,F401
from common.stft import stft, istft                     # noqa: E402

L_WIN, S_HOP = 2048, 512                                 # 第IV部の分離の標準（16 kHz で 128 ms、第7章と同じ）


# ---------------------------------------------------------------- 音源と部屋
def sources(names=("libri1", "libri3"), fs=16000, sec=8.0, offset=0.0):
    """librosa の例の音源を読み、同じ長さにして標準偏差を 1 にそろえる。(N, n) を返す"""
    import librosa
    s = [librosa.load(librosa.ex(nm), sr=fs, mono=True, offset=offset, duration=sec)[0] for nm in names]
    n = min(len(x) for x in s)
    return np.stack([x[:n] / (np.std(x[:n]) + 1e-12) for x in s]).astype(float)


def simulate(src, fs, rt60, seed, n_mics=2, d=0.04, dist=1.5, min_sep_deg=30):
    """直方体の部屋（幅 5〜8 m・奥行 4〜6 m・高さ 3 m）に、間隔 d [m] の直線アレイと音源を置く。
    音源はアレイの中心から dist [m]、方向は互いに min_sep_deg 度以上離す（第7章の simulate と同じ考え方）。
    戻り値：mix (M, n)、img (N, n)（各音源を 0 番のマイクで録った像＝BSS Eval の正解）"""
    import pyroomacoustics as pra
    rng = np.random.default_rng(seed)
    dim = np.array([rng.uniform(5, 8), rng.uniform(4, 6), 3.0])
    if rt60 > 0:
        e_abs, max_order = pra.inverse_sabine(rt60, dim)
        room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(e_abs), max_order=max_order)
    else:
        room = pra.ShoeBox(dim, fs=fs, max_order=0)        # 反射なし（無響）
    c = np.array([dim[0] / 2 + rng.uniform(-0.5, 0.5), dim[1] / 2 + rng.uniform(-0.5, 0.5), 1.5])
    off = (np.arange(n_mics) - (n_mics - 1) / 2) * d
    room.add_microphone_array(np.stack([c + [o, 0, 0] for o in off], axis=1))
    N = src.shape[0]
    while True:
        ang = np.sort(rng.uniform(0, np.pi, N))
        if N == 1 or np.min(np.diff(ang)) >= np.radians(min_sep_deg):
            break
    for k in range(N):
        room.add_source(c + dist * np.array([np.cos(ang[k]), np.sin(ang[k]), 0.0]), signal=src[k])
    pre = room.simulate(return_premix=True)                # (音源, マイク, 時間)
    n = src.shape[1]
    return pre.sum(axis=0)[:, :n], pre[:, 0, :n]


def bin_hz(mask, fs=16000):
    """True のビンの周波数の範囲を "a〜b Hz" の文字列で返す（表示用）"""
    f = np.flatnonzero(mask) * fs / L_WIN
    return f"{f.min():.0f}〜{f.max():.0f} Hz"


def to_stft(x):
    """(M, n) の時間信号 → (F, T, M) の STFT（common.stft、L=2048・S=512・hann）"""
    return stft(np.asarray(x).T, L_WIN, S_HOP)


def to_time(Y, length):
    """(F, T, N) → (N, length) の時間信号（common.istft。length は必ず元の長さ）"""
    return np.stack([istft(Y[:, :, n], L_WIN, S_HOP, length=length) for n in range(Y.shape[2])])


def evaluate(ref, est):
    """BSS Eval（fast_bss_eval）。順番は正解に合わせて選ぶ。音源ごとの SDR・SIR・SAR を返す"""
    import fast_bss_eval
    with np.errstate(divide="ignore", invalid="ignore"):
        sdr, sir, sar, _ = fast_bss_eval.bss_eval_sources(ref, est, compute_permutation=True)
    return np.asarray(sdr), np.asarray(sir), np.asarray(sar)


# ---------------------------------------------------------------- コスト（第7章と同じ式）
def cost_ilrma(W, X, R):
    """Σ_n Σ_ft (|y|²/r + log r) − 2T Σ_f log|det W_f|。R は (N, F, T)"""
    T = X.shape[1]
    P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
    return float(np.sum(P / R + np.log(R)) - 2 * T * logdet(W))


def cost_auxiva(W, X):
    """第7章の AuxIVA のコスト Σ_n Σ_t r_nt − T Σ_f log|det W_f|（r_nt = √Σ_f |y_nft|²）"""
    T = X.shape[1]
    r = np.sqrt(np.sum(np.abs(demix(W, X)) ** 2, axis=0)) + EPS
    return float(np.sum(r) - T * logdet(W))


# ---------------------------------------------------------------- FDICA とパーミュテーション
def fdica(X, n_iter=100, W0=None, floor=1e-3):
    """周波数ごとに独立な ICA。音源モデルは周波数ごとの複素ラプラス分布 p(y) ∝ exp(−2|y|)、
    つまり AuxIVA の r_nt を r_nft = |y_nft| に変えただけ（周波数どうしをつなぐものが何もない）。
    更新は第7章の ip_update（IP）をそのまま使う。
    |y| が 0 に近いと重み 1/|y| が大きくなりすぎて計算が壊れる（NaN）ので、|y| が floor より小さいところは
    floor に切り上げる。これは「r ≤ floor では 2 乗、それより上では r」のフーバー型の G を使うことにあたり、
    G'(r)/r は増えないので補助関数法が使える。floor は反復の間ずっと同じ定数なので、コスト J も1つに決まり、
    J は反復で増えない。IP の正規化で収束後は平均 |y| ≈ 1 になるので、floor = 1e-3 は「平均の 1000 分の 1」の目安。"""
    F, T, M = X.shape
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1)) if W0 is None else W0.copy()
    for _ in range(n_iter):
        for n in range(M):
            r = np.maximum(np.abs(demix(W, X)[:, :, n]), floor)   # (F, T)
            W = ip_update(W, X, 1.0 / r, n)
    return W


def align_oracle(Y, img_stft):
    """正解の像の STFT img_stft (F, T, N) と、振幅の包絡の相関が最大になる順番を周波数ごとに選ぶ（oracle）"""
    import itertools
    F, T, N = Y.shape
    a = np.abs(Y) - np.abs(Y).mean(axis=1, keepdims=True)
    b = np.abs(img_stft) - np.abs(img_stft).mean(axis=1, keepdims=True)
    a /= np.linalg.norm(a, axis=1, keepdims=True) + 1e-12
    b /= np.linalg.norm(b, axis=1, keepdims=True) + 1e-12
    C = np.einsum("ftn,ftk->fnk", a, b)                    # C[f, 出力 n, 音源 k]
    best = np.zeros((F, N), dtype=int)
    score = np.full(F, -np.inf)
    for p in itertools.permutations(range(N)):             # p[k] = 音源 k に割り当てる出力の番号
        s = sum(C[:, p[k], k] for k in range(N))
        better = s > score
        best[better] = p
        score[better] = s[better]
    return best                                            # Y[:, :, best[f]] が音源の順


def align_power_ratio(Yp, n_rounds=10):
    """ブラインドな順番そろえ：プロジェクションバック後の出力 Yp (F, T, N) のパワー比
    q_nft = |y_nft|² / Σ_m |y_mft|² を「その音源がどれだけ鳴っているか」とみなし、
    全周波数の平均（重心）との相関が最大になる順番を周波数ごとに選ぶ、をくり返す。"""
    import itertools
    F, T, N = Yp.shape
    q = np.abs(Yp) ** 2
    q = q / (q.sum(axis=2, keepdims=True) + 1e-12)
    q = q - q.mean(axis=1, keepdims=True)
    q /= np.linalg.norm(q, axis=1, keepdims=True) + 1e-12
    perms = np.tile(np.arange(N), (F, 1))
    for _ in range(n_rounds):
        cur = np.take_along_axis(q, perms[:, None, :], axis=2)
        cen = cur.mean(axis=0)                             # (T, N) 重心
        C = np.einsum("ftn,tk->fnk", q, cen)
        score = np.full(F, -np.inf)
        new = perms.copy()
        for p in itertools.permutations(range(N)):
            s = sum(C[:, p[k], k] for k in range(N))
            better = s > score
            new[better] = p
            score[better] = s[better]
        if np.array_equal(new, perms):
            break
        perms = new
    return perms


def swapped_stats(perms, energy_db, thr_db=-40.0):
    """2 音源のとき、多いほうの順番と違う周波数を数える。
    正解の像のエネルギーが最大から thr_db より小さいビンは、どちらの順番でも差がほとんどなく
    順番が決まらないので数えない。戻り値：(数えたビンでの割合, 入れ替わったビンの数, エネルギーで重みづけた割合)"""
    p = perms[:, 0]
    use = energy_db > thr_db
    maj = np.bincount(p[use]).argmax()
    sw = (p != maj) & use
    w = 10 ** (energy_db / 10)
    return float(sw.sum() / use.sum()), int(sw.sum()), float(np.sum(w[sw]) / np.sum(w))


def energy_db(img_stft):
    """周波数ごとの正解の像のエネルギー [dB]（最大を 0 dB）"""
    E = np.sum(np.abs(img_stft) ** 2, axis=(1, 2))
    return 10 * np.log10(E / E.max() + 1e-30)


def permute(Y, perms):
    """Y (F, T, N) の出力の順番を周波数ごとに perms (F, N) で並べ替える"""
    return np.take_along_axis(Y, perms[:, None, :], axis=2)


# ---------------------------------------------------------------- ILRMA（第7章と同じ計算＋比べるための引数）
def init_ch7(M, F, T, K, seed):
    """第7章の ilrma と同じ初期値（default_rng(seed) で Tb → Vb の順に一様乱数 0.1〜1.0）"""
    rng = np.random.default_rng(seed)
    return rng.uniform(0.1, 1.0, (M, F, K)), rng.uniform(0.1, 1.0, (M, K, T))


def init_pra(M, F, T, K, seed):
    """pyroomacoustics の ilrma と同じ初期値（np.random.seed(seed) のあと rand で T → V の順。V は転置して返す）"""
    rs = np.random.RandomState(seed)
    Tb = 0.1 + 0.9 * rs.rand(M, F, K)
    Vb = 0.1 + 0.9 * rs.rand(M, T, K)
    return Tb, np.ascontiguousarray(np.swapaxes(Vb, 1, 2))


def ilrma_v(X, n_basis=2, n_iter=100, init=None, seed=0, order="ch7", norm="row", fresh_ir=False,
            return_cost=False, checkpoints=(), on_checkpoint=None):
    """ILRMA。order="ch7" かつ norm="row" かつ init=None なら、第7章の bss.ilrma と同じ計算になる。
      init   : (Tb0, Vb0) を与えるとそれを使う（init_ch7・init_pra で作る）
      order  : "ch7"＝全音源の NMF → 全音源の IP、"pra"＝音源ごとに NMF → IP（pyroomacoustics の順。eps も合わせる）
      fresh_ir : order="pra" のとき、各反復の最初の基底の更新で 1/R を作り直す（True）か、
               pyroomacoustics 0.10.1 と同じく正規化の前の 1/R を使う（False）か
      norm   : "row"＝W の n 行目を割る（正しい）、"col"＝W の n 列目に掛ける（pyroomacoustics 0.10.1 の書き方）、
               None＝正規化しない
      checkpoints と on_checkpoint(i, W) で、途中の反復（i 回目のあと）の W を受け取れる
    戻り値：(Y（プロジェクションバック済み）, W, コストの履歴)"""
    F, T, M = X.shape
    Tb, Vb = init if init is not None else init_ch7(M, F, T, n_basis, seed)
    Tb, Vb = Tb.copy(), Vb.copy()
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    hist = []
    if order == "ch7":
        R = Tb @ Vb
    else:
        eps = 1e-15
        R = np.maximum(Tb @ Vb, eps)
        iR_all = 1 / R                                   # pyroomacoustics は 1/R を別の配列に持つ
        P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
    for it in range(1, n_iter + 1):
        if order == "ch7":
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            Vt = np.swapaxes(Vb, 1, 2)
            Tb *= np.sqrt(((P / R ** 2) @ Vt) / ((1 / R) @ Vt + EPS))
            R = Tb @ Vb + EPS
            Tt = np.swapaxes(Tb, 1, 2)
            Vb *= np.sqrt((Tt @ (P / R ** 2)) / (Tt @ (1 / R) + EPS))
            R = Tb @ Vb + EPS
            for n in range(M):
                W = ip_update(W, X, 1.0 / R[n], n)
        else:                                            # pyroomacoustics の順番と eps の扱い
            for s in range(M):
                iR = 1 / R[s] if fresh_ir else iR_all[s]     # pyroomacoustics は正規化のあとの 1/R を作り直さない
                Tb[s] *= np.sqrt(((P[s] * iR ** 2) @ Vb[s].T) / (iR @ Vb[s].T))
                Tb[Tb < eps] = eps
                R[s] = np.maximum(Tb[s] @ Vb[s], eps)
                iR = iR_all[s] = 1 / R[s]
                Vb[s] *= np.sqrt((Tb[s].T @ (P[s] * iR ** 2)) / (Tb[s].T @ iR))
                Vb[Vb < eps] = eps
                R[s] = np.maximum(Tb[s] @ Vb[s], eps)
                iR_all[s] = 1 / R[s]
                U = np.einsum("ftm,ftk,ft->fmk", X, X.conj(), 1 / R[s]) / T
                e = np.zeros((F, M, 1), dtype=complex)
                e[:, s, 0] = 1.0
                w = np.linalg.solve(W @ U, e)[:, :, 0]
                w = w / np.sqrt(np.real(np.einsum("fm,fmk,fk->f", w.conj(), U, w)))[:, None]
                W[:, s, :] = w.conj()
            P = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
        if norm is not None:
            Pn = np.moveaxis(np.abs(demix(W, X)) ** 2, 2, 0)
            lam = np.sqrt(np.mean(Pn, axis=(1, 2))) + (EPS if order == "ch7" else 0.0)   # (N,)
            for s in range(M):
                if norm == "row":
                    W[:, s, :] /= lam[s]
                else:                                    # pyroomacoustics 0.10.1：列に掛けている
                    W[:, :, s] /= lam[s]
                R[s] /= lam[s] ** 2
                Tb[s] /= lam[s] ** 2
                if order != "ch7":
                    P[s] /= lam[s] ** 2
        if return_cost:
            hist.append(cost_ilrma(W, X, R))
        if on_checkpoint is not None and it in checkpoints:
            on_checkpoint(it, W.copy())
    Y = projection_back(demix(W, X), W)
    return Y, W, np.array(hist)


# ---------------------------------------------------------------- oracle（上限の目安）
def oracle_w(X, img_stft):
    """W_f = argmin Σ_t ||s_ft − W x_ft||²（s は正解の像）＝ S Xᴴ (X Xᴴ)⁻¹。出力は最初から像の大きさ"""
    Sxx = np.einsum("ftm,ftk->fmk", X, X.conj())
    Ssx = np.einsum("ftn,ftk->fnk", img_stft, X.conj())
    W = np.linalg.solve(np.swapaxes(Sxx, 1, 2), np.swapaxes(Ssx, 1, 2)).swapaxes(1, 2)   # Ssx Sxx⁻¹
    return W


def ip_oracle(X, img_stft, n_iter=20):
    """音源モデル r_nft に正解の像のパワー |s_nft|² を入れて IP だけを回す（音源モデルが完璧な ILRMA）"""
    F, T, M = X.shape
    R = np.moveaxis(np.abs(img_stft) ** 2, 2, 0) + 1e-6 * np.mean(np.abs(img_stft) ** 2)
    W = np.tile(np.eye(M, dtype=complex), (F, 1, 1))
    for _ in range(n_iter):
        for n in range(M):
            W = ip_update(W, X, 1.0 / R[n], n)
    return projection_back(demix(W, X), W), W

In [ ]:
%%writefile seeds_needed.py
#!/usr/bin/env python3
"""IV-9：「1つだけ変えて比べる」実験に、試行（種・部屋）がいくつ要るかを事前に計算する（対応のある比較）

使い方: cd ~/home-lab/partIV/iv9 && uv run python seeds_needed.py [--sd-diff 1.0] [--delta 0.5]
        予備実験の CSV から差の標準偏差を読むとき：
        uv run python seeds_needed.py --pilot out/repro_window.csv --a ILRMA --b AuxIVA --where win_ms=128 --delta 0.5
考え方（本文 IV-9-3-6）
  試行 i で、新しい方法 A と元の方法 B の SDR 改善量の差 d_i = A_i − B_i をとる（同じ部屋・同じ音源・同じ初期値）。
  d の平均 δ、標準偏差 σ_d とすると、両側の有意水準 α・検出力 1−β の t 検定に要る試行の数は
    正規近似：n ≈ ((z_{1−α/2} + z_{1−β}) σ_d / δ)²
    正確な値：非心 t 分布で検出力を計算し、1−β 以上になる最小の n
  を自分で計算し、statsmodels の TTestPower と照合する（差は 0 か、丸めの 1 以内のはず）。
"""
import argparse

import numpy as np
from scipy import stats


def n_normal(delta, sd, alpha=0.05, power=0.8):
    """正規近似の試行の数（切り上げ前の実数も返す）"""
    z = stats.norm.ppf(1 - alpha / 2) + stats.norm.ppf(power)
    x = (z * sd / delta) ** 2
    return int(np.ceil(x)), x


def power_t(n, delta, sd, alpha=0.05):
    """対応のある t 検定（両側）の検出力。T = mean(d)/(s/√n) は自由度 n−1、非心度 δ√n/σ_d の非心 t 分布に従う"""
    df = n - 1
    nc = delta * np.sqrt(n) / sd
    c = stats.t.ppf(1 - alpha / 2, df)
    p = stats.nct.sf(c, df, nc) + stats.nct.cdf(-c, df, nc)
    return 1.0 if np.isnan(p) and nc > 10 else float(p)   # 非心度がとても大きいと scipy が nan を返すことがある（検出力はほぼ 1）


def n_exact(delta, sd, alpha=0.05, power=0.8, n_max=10000):
    """検出力が power 以上になる最小の n（n ≥ 2）"""
    for n in range(2, n_max + 1):
        if power_t(n, delta, sd, alpha) >= power:
            return n
    raise ValueError("n_max までに届かない")


def mde(n, sd, alpha=0.05, power=0.8):
    """n 試行で、検出力 power で見つけられる最小の差（正確な t。二分法）"""
    lo, hi = 1e-6, 10 * sd
    for _ in range(100):
        mid = (lo + hi) / 2
        if power_t(n, mid, sd, alpha) >= power:
            hi = mid
        else:
            lo = mid
    return hi


def sd_diff(sd_a, sd_b, rho):
    """差の標準偏差 σ_d = √(σ_A² + σ_B² − 2ρσ_Aσ_B)：同じ試行の A と B の相関 ρ が大きいほど小さい"""
    return float(np.sqrt(sd_a ** 2 + sd_b ** 2 - 2 * rho * sd_a * sd_b))


def from_pilot(path, a, b, where):
    import pandas as pd
    df = pd.read_csv(path)
    for cond in where or []:
        k, v = cond.split("=")
        df = df[df[k].astype(str) == v] if not np.issubdtype(df[k].dtype, np.number) else df[df[k] == float(v)]
    p = df.pivot_table(index="seed", columns="method", values="sdr_imp")
    d = (p[a] - p[b]).dropna()
    r = np.corrcoef(p[a], p[b])[0, 1]
    print(f"予備実験：{path}（{' '.join(where or [])}）、{a} − {b}、試行 {len(d)}")
    print(f"  σ_{a} = {p[a].std(ddof=1):.2f} dB、σ_{b} = {p[b].std(ddof=1):.2f} dB、相関 ρ = {r:.2f}、"
          f"差の平均 {d.mean():+.2f} dB、σ_d = {d.std(ddof=1):.2f} dB")
    return float(d.std(ddof=1)), len(d)


def sd_interval(sd, n, level=0.95):
    """標準偏差の区間（正規分布を仮定）：σ ∈ [s√((n−1)/χ²_{1−a/2}), s√((n−1)/χ²_{a/2})]"""
    a = 1 - level
    return (sd * np.sqrt((n - 1) / stats.chi2.ppf(1 - a / 2, n - 1)),
            sd * np.sqrt((n - 1) / stats.chi2.ppf(a / 2, n - 1)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sd-diff", type=float, default=1.0, help="差の標準偏差 σ_d [dB]")
    ap.add_argument("--delta", type=float, default=0.5, help="見つけたい差 δ [dB]（実用上意味のある最小の差）")
    ap.add_argument("--alpha", type=float, default=0.05)
    ap.add_argument("--power", type=float, default=0.8)
    ap.add_argument("--pilot", help="予備実験の CSV（列 seed・method・sdr_imp）")
    ap.add_argument("--a", default="ILRMA")
    ap.add_argument("--b", default="AuxIVA")
    ap.add_argument("--where", nargs="*", help="CSV を絞る条件（例：win_ms=128）")
    a = ap.parse_args()
    if a.pilot:
        sd, n_pilot = from_pilot(a.pilot, a.a, a.b, a.where)
        lo, hi = sd_interval(sd, n_pilot)
        print(f"  σ_d の 95% 区間（正規分布を仮定）[{lo:.2f}, {hi:.2f}] dB → 要る試行の数は "
              f"{n_exact(a.delta, lo, a.alpha, a.power)}〜{n_exact(a.delta, hi, a.alpha, a.power)}"
              f"（予備実験が小さいと σ_d の推定も不確か。上側の値で見積もるか、途中で σ_d を見直す）")
    else:
        sd = a.sd_diff
    print("\n[1] 差の標準偏差は、対応をとると小さくなる（σ_A = σ_B = 2 dB の例）")
    for rho in (0.0, 0.5, 0.9, 0.97):
        print(f"  ρ = {rho:4.2f} → σ_d = {sd_diff(2, 2, rho):.3f} dB")

    print(f"\n[2] δ = {a.delta} dB を、α = {a.alpha}（両側）・検出力 {a.power} で見つけるのに要る試行の数（σ_d = {sd:.2f} dB）")
    nn, x = n_normal(a.delta, sd, a.alpha, a.power)
    ne = n_exact(a.delta, sd, a.alpha, a.power)
    from statsmodels.stats.power import TTestPower
    sm = TTestPower().solve_power(effect_size=a.delta / sd, alpha=a.alpha, power=a.power, alternative="two-sided")
    print(f"  正規近似 {x:.2f} → {nn}、非心 t（自前）{ne}、statsmodels {sm:.2f} → {int(np.ceil(sm))}")
    assert abs(ne - int(np.ceil(sm))) <= 1, "自前の計算と statsmodels が 1 より大きく違う"
    print(f"  自前の n = {ne} での検出力 {power_t(ne, a.delta, sd, a.alpha):.4f}、"
          f"statsmodels の検出力 {TTestPower().power(a.delta / sd, ne, a.alpha):.4f}")

    print(f"\n[3] 試行 n 個で見つけられる最小の差（検出力 {a.power}）")
    for n in (5, 10, 20, 40):
        m = mde(n, sd, a.alpha, a.power)
        z = (stats.norm.ppf(1 - a.alpha / 2) + stats.norm.ppf(a.power)) * sd / np.sqrt(n)
        e = TTestPower().solve_power(nobs=n, alpha=a.alpha, power=a.power, alternative="two-sided") * sd
        print(f"  n = {n:2d}：正確な t {m:.3f} dB（statsmodels {e:.3f} dB、正規近似 {z:.3f} dB）")

    print("\n[4] ウィルコクソンの符号順位検定（両側）で p < 0.05 になりうる最小の試行の数：p の最小値 2/2^n")
    for n in (5, 6, 7):
        print(f"  n = {n}：{2 / 2 ** n:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile theme_sheet.py
#!/usr/bin/env python3
"""IV-9：研究テーマの候補のワークシートを Markdown で作り、抜けと「近すぎ・遠すぎ」を点検する

使い方: cd ~/home-lab/partIV/iv9 && uv run python theme_sheet.py            # 記入見本 3 つ → out/theme_sheet.md
        uv run python theme_sheet.py --blank                                  # 空欄のひな形 → out/theme_sheet_blank.md
自分の候補を書くときは、CANDIDATES の 3 つを書き換える（見本の文をそのまま残さない）。
  ・why（なぜこのテーマで）の各項目は ("事実", "…出典…") か ("推測", "…") にする。事実には必ず出典（URL・論文・題目）を書く。
  ・score は CRITERIA の 6 つの観点を 0〜2 で自己採点する。ルールで「近すぎ」「遠すぎ」「有力」を判定する。
研究室訪問の前日に、out/theme_sheet.md を印刷するか PDF にして持っていく。
"""
import argparse
import datetime
import pathlib

OUT = pathlib.Path("out").resolve()   # ノートブック版：このノートブックの隣の out/
OUT.mkdir(exist_ok=True)

CRITERIA = {
    "Q": "問いが 1 文で書け、答えが数値か「はい／いいえ」で出る",
    "LAB": "音響信号処理の研究・資源とつながる（論文・題目・装置を 1 つ以上、出典つきで指せる）",
    "NEW": "既存の研究（研究室の題目・論文）と 1 文で区別できる（やり直しではない）",
    "M1": "最初の 1 か月で、家の機材とシミュレーションだけで予備的な結果が出る",
    "ME": "自分の強み（統計・電子計測・CWT の異音検知の経験）が効く",
    "ALT": "失敗しても学べることがあり、代わりの案がある",
}

CANDIDATES = [
    dict(
        title="① 非同期録音のサンプリング周波数のずれ（SFO）の補正と分離（最終演習K の延長）",
        question="分散した録音機（PC・スマホ）の SFO を合図なしで推定・補正するとき、ILRMA の SDR 改善量を"
                 "同期録音との差 0.5 dB 以内に保つには、推定の誤差を何 ppm 以内にすればよいか。"
                 "合図なしの推定は、残響時間 0.3 s・区間 8 秒でその精度に届くか。",
        why=[("事実", "分散マイクアレイのスポットフォーミング（Ayano, Li, Seki, Kitamura, EUSIPCO 2024, "
                     "doi:10.23919/EUSIPCO63174.2024.10714995）。実験は「All microphones were synchronized」の仮定"),
             ("事実", "この分野では、時計のそろっていない複数のアレイで録ったデータを集める研究もある"),
             ("事実", "複数のアレイを BSS に使う研究の方向もある"),
             ("推測", "SFO の推定・補正は、上の研究が前提として済ませている前処理にあたる。音源分離の論文で SFO そのものを"
                     "題にしたものは見つからなかった（執筆時に業績一覧と題目一覧で確認）ので、重なりは小さい")],
        prior=["N. Ono ほか, WASPAA 2009（非同期録音のブラインドな位置合わせ）",
               "S. Miyabe, N. Ono, S. Makino, Signal Processing 107, 2015（最尤推定による SFO の補正）",
               "L. Wang, S. Doclo, IEEE/ACM TASLP 24(3), 2016（相関の最大化による SFO の推定）"],
        prereq="IV-2（STFT の位相）、IV-5-3-10（SFO と位相の回転）、IV-6（ILRMA）、最終演習K（clockfix・drift_search）、"
               "II-4・II-5（時刻合わせ）",
        tools="家：Python（第7章の bss.py、最終演習K のコード）、pyroomacoustics、Mac とスマホ。"
              "研究室（訪問で確かめる）：同期マイクアレイで録り、一部のチャネルに既知のずれを後から加える",
        strengths="統計（同等性の検定・混合モデルで「許容誤差」を区間つきで出す）、電子計測（水晶のずれ・時刻合わせ）",
        plan=["週1：最終演習K の結果を見直す。Miyabe 2015・Wang 2016 を 3 段階で読み、ワークシートを 2 枚書く。"
              "EUSIPCO 2024 の実験条件を表にする",
              "週2：事前登録を Git にコミットしてから、シミュレーション（ずれ 0・10・30・100 ppm × 補正の誤差 0・0.1・1・3 ppm、"
              "種 10 通り、T60 0・0.256 s）で SDR の低下を測る",
              "週3：合図なしの推定（drift_search）の誤差の分布を、区間の長さ（4・8・16 秒）と残響で測り、週2 の許容誤差と比べる",
              "週4：A4 2 枚と図 3 枚にまとめる。研究室訪問で見せ、データセットの録音で SFO をどう扱ったかを質問する"],
        risks=["研究室では SFO をすでに処理済み、または扱わない方針かもしれない→ 訪問で確かめる",
               "既存の推定法の再実装で終わり、新しさが出ない → 貢献を「分離性能から逆算した許容誤差の基準」に絞る"],
        alternative="SFO ではなく、アレイ間の開始時刻のずれだけ、またはアレイの数を減らしたときのスポットフォーミングの性能に絞る。"
                    "評価の部分は候補③に合流できる",
        score=dict(Q=2, LAB=2, NEW=1, M1=2, ME=2, ALT=2),
    ),
    dict(
        title="② ドラムの被り音の抑圧に、振動センサの信号を参照として使う（電子工作の技能を活かす）",
        question="太鼓に取り付けた振動センサ（圧電素子）の「被りのない」信号を、NMF の活性化（いつ叩いたか）の手がかりに使うと、"
                 "近接マイクの録音の被り音（SIR）は、センサなしの NMF より何 dB 良くなるか。",
        why=[("事実", "この分野では、被り音の抑圧が続けて研究されている（時間チャネル NMF、音量比特徴量、"
                     "セミブラインド ILRMA、深層学習）"),
             ("事実", "Y. Mizobuchi, D. Kitamura ほか, \"Music bleeding-sound reduction based on time-channel nonnegative "
                     "matrix factorization,\" APSIPA Trans. Signal Inf. Process. 14(1), e18, 2025, doi:10.1561/116.20250020"),
             ("事実", "被り音の低減で、近接マイクの音を手がかりに使う研究もある"),
             ("推測", "センサの信号は「被りの少ないチャネル」の一種とみなせる。音だけで解く研究室の流れに、別の種類の入力を足す形")],
        prior=["Mizobuchi ほか, APSIPA Trans. 2025（時間チャネル NMF）",
               "国内の学会発表（日本音響学会の研究発表会の予稿で各自が探す）",
               "録音の現場で使われるドラムトリガ（圧電素子）とゲート（分離の研究での先行例は週1で探す）"],
        prereq="第7章（NMF）、IV-8（楽音・音楽の NMF）、IV-2、第1章（ADC）、II-5（標本化）、最終演習K（2 つの記録の時刻合わせ）",
        tools="家：Pico（ADC）、圧電素子 1〜2 個（任意で追加）、保護回路（並列の放電抵抗、大きめの直列抵抗、1.65 V へのバイアス、"
              "1N5819 のクランプ。値はデータシートで要確認。本文 3-9 の ⚡）、"
              "Mac の録音、練習パッドか机（ドラムが無ければ）。研究室：マルチトラックレコーダ、"
              "公開されているドラムのデータセット（利用条件は公開先で要確認）",
        strengths="電子工作（センサと ADC、保護回路）、時刻合わせ、CWT（打撃の立ち上がりの検出）",
        plan=["週1：被り音の論文 3 本を読み（Mizobuchi 2025 は 3 段階すべて）、公開データセットの構成と利用条件を確かめる。"
              "ドラムトリガ（圧電素子）の信号を分離・被り音の抑圧に使った先行例を探す",
              "週2：センサを買う前に、公開データの近接マイクの信号を「センサ風」（帯域を狭め、ひずませる）にして参照に使う"
              "シミュレーションで、効果の上限を見積もる（oracle の段階）",
              "週3：圧電素子 1 個の最大の電圧を、まず分圧してマルチメータで確かめる。そのあと保護回路つきで Pico の ADC につなぎ、"
              "打撃を記録する。マイク録音との時刻合わせの精度（ms）を測る",
              "週4：同じ録音で、参照あり NMF と参照なし NMF を比べ（試行＝打撃のフレーズ、10 フレーズ以上）、まとめる"],
        risks=["センサの信号は音と周波数特性が違う → 使うのは活性化（時刻と強さ）だけにする",
               "Pico の ADC（有効 約 9 ビット）と標本化の速さが足りない可能性 → 包絡だけを記録する",
               "研究室の被り音の研究は「マイクだけで解く」ことが前提かもしれない → 訪問で確かめる",
               "被り音の研究は共同研究のテーマかもしれない（推測）→ 学生が加わるときの約束"
               "（データ・成果の公開、知財）を訪問で確かめる"],
        alternative="センサを使わず、公開データセットで「被りの少ないチャネル（近接マイク）の使い方」を比べる研究に絞る",
        score=dict(Q=2, LAB=2, NEW=1, M1=1, ME=2, ALT=2),
    ),
    dict(
        title="③ 分離の評価の統計：種と部屋のばらつきを分けて、手法の差を正しく区間推定する",
        question="ILRMA と AuxIVA の SDR 改善量の差のばらつきのうち、初期値（種）による分と部屋・配置による分はそれぞれ何割か。"
                 "差の 95% 区間の幅を 1 dB 以下にするには、部屋の数 R と種の数 S をどう配分すればよいか。",
        why=[("事実", "音源分離の論文の評価の作法：乱数の種 10 通りの平均と標準偏差（EUSIPCO 2024、arXiv:2407.08951）"),
             ("事実", "この分野では、評価や性能の性質そのもの（指標どうしの関係など）"
                     "を調べた研究もある"),
             ("事実", "この分野には、評価の統計の扱いを深める研究もある"),
             ("推測", "手法を提案する研究ではないので、単独のテーマとしてよりも、ほかのテーマの評価の部分として生きる可能性が高い")],
        prior=["E. Vincent, R. Gribonval, C. Févotte, IEEE TASLP 14(4), 2006（BSS Eval）",
               "J. Le Roux ほか, ICASSP 2019（SDR – half-baked or well done?）",
               "最終演習K の K-4-3（種の数と試行の数を分散の成分から決める）"],
        prereq="第7章 7-3-4、IV-6（eval_iv6.py）、IV-7（評価指標）、最終演習K（K-4-3）、第1〜3章（混合モデル・ベイズ）",
        tools="家：Python（statsmodels の MixedLM、PyMC）、IV-6 の評価のコード。計算が重いときは第III部のクラウド。"
              "研究室：計算機（クラスタなど）で条件を大きく増やす",
        strengths="統計（統計検定 2 級・準 1 級の分散分析・混合モデル・ブートストラップ・ベイズ）",
        plan=["週1：音源分離の論文 5 本の「評価の書き方」（何を平均したか、SD は何のばらつきか、検定の有無）を表にする",
              "週2：部屋 R = 10 × 初期値の種 S = 5 の入れ子の計画で ILRMA と AuxIVA を回す。IV-6・この章のコードは 1 つの種で"
              "部屋と初期値を同時に決めているので、部屋の種と初期値の種を別の引数に分けるよう直してから（事前登録してから）",
              "週3：線形混合モデルで分散の成分を推定し、R と S の配分ごとの区間の幅を計算する。ブートストラップの区間とも比べる",
              "週4：報告の仕方の提案（平均 ± SD と、差の区間の両方を書く）をまとめ、候補①②の評価の計画に組み込む"],
        risks=["研究室の卒研として「手法の提案」が求められるなら、単独では弱い → ①か②の評価の章として組み込む",
               "計算量（R × S × 手法 × 条件）が大きい → 音を短くし、条件を絞る"],
        alternative="評価の統計を、候補①（許容誤差の区間推定）か②（打撃のフレーズを単位にした比較）の一部にする",
        score=dict(Q=2, LAB=1, NEW=2, M1=2, ME=2, ALT=1),
    ),
]


def judge(score):
    total = sum(score.values())
    if score["LAB"] == 0:
        return total, "遠すぎ（音響信号処理の研究・資源とつながらない）"
    if score["NEW"] == 0:
        return total, "近すぎ（既存の研究のやり直しに近い）"
    if min(score.values()) == 0:
        return total, "要検討（0 点の観点がある）"
    return total, ("有力" if total >= 10 else "候補（弱い観点を補う）")


def render(cands, blank=False):
    today = datetime.date.today().isoformat()
    L = [f"# 研究テーマの候補ワークシート（{today}）", "",
         "書き方：事実には出典を、推測には「推測」と書く。採点は 0（当てはまらない）〜2（よく当てはまる）。", ""]
    for c in cands:
        L += [f"## {c['title']}", "", f"**問い（1 文）**：{c['question']}", "", "**なぜこのテーマで**", ""]
        L += [f"- （{k}）{v}" for k, v in c["why"]] + [""]
        L += ["**先行研究**", ""] + [f"- {p}" for p in c["prior"]] + [""]
        L += [f"**必要な前提（この本の章）**：{c['prereq']}", "", f"**道具**：{c['tools']}", "",
              f"**自分の強みが効くところ**：{c['strengths']}", "", "**最初の 1 か月**", ""]
        L += [f"{i + 1}. {p}" for i, p in enumerate(c["plan"])] + [""]
        L += ["**リスク**", ""] + [f"- {r}" for r in c["risks"]] + [""]
        L += [f"**代わりの案**：{c['alternative']}", "", "**自己採点**", "", "| 観点 | 点 |", "|---|---|"]
        L += [f"| {CRITERIA[k]} | {'' if blank else c['score'][k]} |" for k in CRITERIA]
        if not blank:
            total, verdict = judge(c["score"])
            L += ["", f"**合計 {total} / 12：{verdict}**"]
        L += [""]
    return "\n".join(L)


def check(c):
    """記入の抜けを点検する（空欄、事実に出典がない、計画が 4 週でない、など）"""
    probs = []
    for k in ("title", "question", "prereq", "tools", "strengths", "alternative"):
        if not c.get(k):
            probs.append(f"{k} が空")
    if not any(k == "事実" for k, _ in c["why"]):
        probs.append("「なぜこのテーマで」に事実（出典つき）が 1 つもない")
    for k, v in c["why"]:
        if k == "事実" and not any(s in v for s in ("http", "doi", "arXiv", "題目", "卒研", "特研", "科研", "EUSIPCO", "APSIPA")):
            probs.append(f"事実なのに出典が見当たらない：{v[:30]}…")
        if k not in ("事実", "推測"):
            probs.append(f"why の種類は「事実」か「推測」：{k}")
    if len(c["plan"]) != 4:
        probs.append(f"1 か月の計画が {len(c['plan'])} 週（4 週で書く）")
    if len(c["prior"]) < 2:
        probs.append("先行研究が 2 本未満")
    if set(c["score"]) != set(CRITERIA) or not all(0 <= v <= 2 for v in c["score"].values()):
        probs.append("採点は 6 つの観点すべてを 0〜2 で")
    return probs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--blank", action="store_true", help="空欄のひな形を作る")
    a = ap.parse_args()
    if a.blank:
        empty = dict(title="（候補の名前）", question="", why=[("事実", ""), ("推測", "")], prior=["", ""], prereq="",
                     tools="", strengths="", plan=["週1：", "週2：", "週3：", "週4："], risks=[""], alternative="",
                     score={k: 0 for k in CRITERIA})
        path = OUT / "theme_sheet_blank.md"
        path.write_text(render([dict(empty, title=f"候補{i}") for i in "①②③"], blank=True), encoding="utf-8")
        print(f"{path} を保存しました（空欄のひな形）")
        return
    ok = True
    for c in CANDIDATES:
        total, verdict = judge(c["score"])
        probs = check(c)
        ok &= not probs
        print(f"{c['title'][:40]}…  合計 {total}/12 → {verdict}")
        for p in probs:
            print("   点検：", p)
    path = OUT / "theme_sheet.md"
    path.write_text(render(CANDIDATES), encoding="utf-8")
    print(f"{path} を保存しました" + ("" if ok else "（点検で見つかった抜けを直してから使う）"))


if __name__ == "__main__":
    main()

In [ ]:
import sys, re, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
warnings.filterwarnings("ignore", message="Glyph")
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common import plotting                      # import すると Hiragino Sans が設定される
import bss6, bss                                 # bss6 は第7章の bss.py を読み込む
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
print("IV-6 と第7章を読めた：", bss.__file__)


def listen_file(path):
    """保存した wav をそのままの大きさで再生ボタンにする。normalize=False を消さない"""
    import numpy as np, soundfile   # 各節のプログラムが np などの名前を使うので、ここで読み直す
    x, fs = soundfile.read(path)
    assert np.max(np.abs(x)) <= 0.32, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1, text=None):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout if text is None else text)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが ok などの名前を使うことがあるので、確かめのセルで戻す

## 1. 研究地図を作る（IV-9-3-2・3-3、手順 A、`thesis_map.py`）

説明のための架空の見本の題目 15 件（6 年分）を、キーワードの規則でタグに分け、件数と時期ごとの割合を出します（数秒）。分類はこの本の判断で、研究室の公式の分類ではありません。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['thesis_map.py']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
# 教科書：uv run python thesis_map.py
#!/usr/bin/env python3
"""IV-9：卒業研究・特別研究の題目を分類して、研究地図の数を出す（見本の題目 15 件）

使い方: cd ~/home-lab/partIV/iv9 && uv run python thesis_map.py
題目：説明のための架空の見本 15 件（6 年分、本科の卒業研究 11 件・専攻科の特別研究 4 件）。
  年度は 1〜6 年目の番号で表す。実際の研究室の題目は、配属先の研究室のページの一覧で調べ、TITLES を書き換える
  （件数の assert も直す）。学生の名前は入れない。
分類はこの本の判断（キーワードの規則）で、研究室の公式の分類ではない。1 つの題目に複数のタグがつく。
出力：out/thesis_map.csv（題目ごとのタグ）、out/thesis_map.png（年ごとのタグの数）、画面に集計
"""
import pathlib
import re

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]   # 既存の章と同じ日本語フォント
matplotlib.rcParams["axes.unicode_minus"] = False

OUT = pathlib.Path(__file__).resolve().parent / "out"
OUT.mkdir(exist_ok=True)

# (年目, 種別, 題目)。種別：卒研＝本科の卒業研究、特研＝専攻科の特別研究。説明のための架空の見本
TITLES = [
    (1, "卒研", "NMF による楽器音の分離と基底の数の選び方"),
    (1, "卒研", "独立ベクトル分析によるブラインド音源分離の反復回数の検討"),
    (2, "卒研", "補助関数法による非負値行列因子分解の高速化"),
    (2, "特研", "残響のある部屋での独立低ランク行列分析の窓長の選び方"),
    (2, "卒研", "スマートフォンで録った会話の発話区間の検出"),
    (3, "卒研", "講義の録音の雑音を、ブラインド音源分離で減らす"),
    (3, "特研", "STFT の無矛盾性を使った位相の推定"),
    (4, "卒研", "深層学習による環境音の識別と、少ないデータでの学習"),
    (4, "卒研", "マイクロホンアレイによる話者の方向の推定"),
    (4, "特研", "周波数ごとの順番（パーミュテーション）を深層学習でそろえる音源分離"),
    (5, "卒研", "ピアノの演奏の自動採譜と音楽の NMF"),
    (5, "卒研", "音の波形を見せる教材のシステム"),
    (6, "特研", "スマートウォッチの心拍のデータの解析"),
    (6, "卒研", "電話の音声を聞きやすくする深層学習"),
    (6, "卒研", "加速度センサによる歩行の解析"),
]

# タグとキーワード（正規表現）。上から順に「主な分類」を決めるときの優先順位にもなる
TAGS = {
    "生体信号": r"心拍",
    "アレイ・空間": r"マイクロホンアレイ|アレイ",
    "音楽": r"被り音|ドラム|ギター|楽器|音楽|ピアノ",
    "分離の数理": r"非負値|NMF|独立低ランク|独立ベクトル|ブラインド|音源分離|パーミュテーション|無矛盾",
    "深層学習": r"深層|DNN|ニューラル",
    "音声・表現": r"発話|帯域拡張|音声",
    "システム・データ": r"システム|データセット",
}


def tag(title):
    return [k for k, pat in TAGS.items() if re.search(pat, title)]


def main():
    df = pd.DataFrame(TITLES, columns=["year", "kind", "title"])
    assert len(df) == 15 and (df.kind == "卒研").sum() == 11 and (df.kind == "特研").sum() == 4, "件数が見本と違う"
    df["tags"] = df.title.map(tag)
    df["main"] = df.tags.map(lambda t: t[0] if t else "その他")
    for k in TAGS:
        df[k] = df.tags.map(lambda t, k=k: k in t)
    df.drop(columns="tags").to_csv(OUT / "thesis_map.csv", index=False)

    print(f"題目 {len(df)} 件（卒研 {(df.kind == '卒研').sum()}・特研 {(df.kind == '特研').sum()}）、"
          f"{df.year.min()}〜{df.year.max()} 年目")
    print("\nタグごとの件数（1 件に複数のタグがつくので、合計は題目の数を超える）")
    for k in TAGS:
        ex = df.loc[df[k], "title"]
        print(f"  {k:10s} {df[k].sum():2d} 件  例：{ex.iloc[0] if len(ex) else '—'}")
    print(f"\n主な分類（優先順位：{' > '.join(TAGS)}）")
    print(df.main.value_counts().to_string())
    none = df[df.tags.map(len) == 0]
    if len(none):
        print("\nどのタグにも当たらない題目（規則を見直すか「その他」として読む）：")
        for t in none.title:
            print("  ", t)
    # 時期による変化：1〜3 年目と 4〜6 年目に分けて、タグの割合を比べる
    df["period"] = (df.year >= 4).map({False: "1〜3年目", True: "4〜6年目"})
    share = df.groupby("period")[list(TAGS)].mean().T * 100
    cnt = df.groupby("period").size()
    print("\n時期ごとのタグの割合 [%]（件数：" + "、".join(f"{p} {n} 件" for p, n in cnt.items()) + "）")
    print(share.round(0).astype(int).to_string())
    print("※ 件数が少ないので、割合の差は「傾向の手がかり」にとどめる（検定はしない。下の確認問題を参照）")

    years = list(range(df.year.min(), df.year.max() + 1))
    fig, ax = plt.subplots(figsize=(8, 4))
    bottom = pd.Series(0, index=years)
    for k in TAGS:
        v = df[df[k]].groupby("year").size().reindex(years, fill_value=0)
        ax.bar(years, v, bottom=bottom, label=k)
        bottom += v
    ax.set_xlabel("年目")
    ax.set_ylabel("タグの数（1 件に複数）")
    ax.set_title("卒研・特研の題目のタグ（見本。この本の分類）")
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(OUT / "thesis_map.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("\nout/thesis_map.csv と out/thesis_map.png を保存しました")


if __name__ == "__main__":
    main()


In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
ok("題目 15 件（卒研 11・特研 4）" in cap.stdout, "「題目 15 件（卒研 11・特研 4）」")
tags = {k: int(v) for k, v in re.findall(r"^  (\S+)\s+(\d+) 件  例", cap.stdout, re.M)}
want = {"分離の数理": 8, "深層学習": 3, "音楽": 2, "音声・表現": 2, "生体信号": 1, "アレイ・空間": 1, "システム・データ": 1}
ok(tags == want, f"タグの件数が 分離の数理 8・深層学習 3・音楽 2・音声・表現 2・生体信号 1・アレイ・空間 1・システム・データ 1（{tags}）")
none = cap.stdout.split("どのタグにも当たらない題目")[1].split("時期ごと")[0].strip().splitlines()[1:] if "どのタグにも当たらない題目" in cap.stdout else []
ok(len([l for l in none if l.strip()]) == 1, "どのタグにも当たらない題目が 1 件")
import pandas as pd
d = pd.read_csv("out/thesis_map.csv")
ok(d["main"].value_counts().to_dict() == {"分離の数理": 6, "音楽": 2, "深層学習": 2, "音声・表現": 1, "アレイ・空間": 1, "システム・データ": 1, "生体信号": 1, "その他": 1},
   "主な分類が 分離の数理 6・音楽 2・深層学習 2・ほか各 1（7 節の結果の例）")
p = d.assign(late=d.year >= 4).groupby("late")["分離の数理"].mean() * 100
ok(round(p[False]) == 86 and round(p[True]) == 25, f"時期ごとの分離の数理の割合 86% → 25%（{p[False]:.0f}% → {p[True]:.0f}%）")


**規則を自分で直してみる**（手順 A の 3。このセルはノートブックで足したもの）：「生体信号」のキーワードに「センサ」を足して、もう一度集計します。分類は判断で決まることを確かめてください。終わったら、上のセルを実行し直すと元の規則に戻ります。

In [ ]:
%%capture cap
# ノートブック版で足したセル：上のセルで定義した TAGS を 1 か所だけ書きかえて main() をもう一度呼ぶ（教科書では thesis_map.py を直接書きかえる）
TAGS["生体信号"] = TAGS["生体信号"] + "|センサ"
print("生体信号 の規則：", TAGS["生体信号"])
main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
import re as _re
n_bio = int(_re.search(r"^  生体信号\s+(\d+) 件", cap.stdout, _re.M).group(1))
ok(n_bio == 2, f"「センサ」を足すと、生体信号が 1 → 2 件になる（{n_bio} 件）")
import pandas as pd
d = pd.read_csv("out/thesis_map.csv")
ok(bool(d.loc[d.title == "加速度センサによる歩行の解析", "生体信号"].iloc[0]),
   "入ったのは見本の「加速度センサによる歩行の解析」")


## 2. 書誌を確かめる（IV-9-3-4、手順 B の 2、`crossref_check.py`）

DOI から Crossref の公開 API で、題・誌名・巻・号・ページ・年を取ってきます（1 件 1 秒程度）。**インターネットが必要**です。User-Agent には個人情報を入れません（この本の約束）。つながらないときは「取得できません」と表示され、下の確かめは「つながらなかった」とだけ表示して先へ進みます。

`my_refs.csv` は 6 節の「自分のメモ」の例です。窓長の論文のページを、PDF の印刷のとおり 1210-1214 と書いてあります。

In [ ]:
%%writefile my_refs.csv
doi,title,pages,year
10.23919/EUSIPCO.2017.8081392,,1210-1214,2017
10.1109/TASLP.2016.2577880,,1626-1641,2016

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['crossref_check.py', '10.1109/TASLP.2016.2577880', '10.23919/EUSIPCO.2017.8081392', '10.1587/transinf.2020EDK0002']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
# 教科書：uv run python crossref_check.py 10.1109/TASLP.2016.2577880 10.23919/EUSIPCO.2017.8081392 10.1587/transinf.2020EDK0002
#!/usr/bin/env python3
"""IV-9：論文の書誌情報（題・誌名・巻・号・ページ・年）を、DOI から Crossref で確かめる

使い方: cd ~/home-lab/partIV/iv9 && uv run python crossref_check.py 10.1109/TASLP.2016.2577880 10.23919/EUSIPCO.2017.8081392
        自分のメモと照合するとき（CSV の列：doi,title,pages,year。空欄は照合しない）：
        uv run python crossref_check.py --csv my_refs.csv
Crossref の公開 API（https://api.crossref.org/works/<DOI>）を読むだけで、登録は要らない。
  ・インターネットにつながっている必要がある。1 秒に 1 回程度に抑える（下の SLEEP）。
  ・User-Agent には自分のメールアドレスなどの個人情報を入れない（この本の約束）。
  ・Crossref の値が正しいとは限らない（会議録のページ番号が PDF の印刷と違う例がある。本文 IV-9-3-4）。
    違ったら、出版社のページ・PDF も見て、どれに従ったかを自分のメモに書く。
"""
import argparse
import csv
import json
import time
import urllib.parse
import urllib.request

API = "https://api.crossref.org/works/"
UA = "home-lab-textbook-refcheck/1.0 (python urllib)"
SLEEP = 1.0


def fetch(doi):
    url = API + urllib.parse.quote(doi.strip())
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)["message"]


def summary(m):
    first = lambda v: (v or [""])[0]                                   # noqa: E731
    year = (m.get("published") or m.get("issued") or {}).get("date-parts", [[None]])[0][0]
    authors = ", ".join(f"{a.get('given', '')} {a.get('family', '')}".strip() for a in m.get("author", []))
    return dict(doi=m.get("DOI", ""), title=first(m.get("title")), container=first(m.get("container-title")),
                volume=m.get("volume", ""), issue=m.get("issue", ""), pages=m.get("page", ""),
                year=year, authors=authors)


def norm(s):
    return " ".join(str(s).lower().replace("–", "-").replace("—", "-").split())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("dois", nargs="*")
    ap.add_argument("--csv", help="照合したいメモ（列 doi,title,pages,year）")
    a = ap.parse_args()
    rows = [dict(doi=d) for d in a.dois]
    if a.csv:
        with open(a.csv, newline="", encoding="utf-8") as f:
            rows += list(csv.DictReader(f))
    if not rows:
        ap.error("DOI か --csv を与えてください")
    n_bad = 0
    for i, row in enumerate(rows):
        if i:
            time.sleep(SLEEP)
        try:
            s = summary(fetch(row["doi"]))
        except Exception as e:                                         # 見つからない DOI、通信の失敗など
            print(f"\n{row['doi']}：取得できません（{e}）。DOI の打ち間違いか、Crossref 以外（DataCite など）の DOI")
            n_bad += 1
            continue
        print(f"\n{s['doi']}\n  {s['authors']}\n  \"{s['title']}\"\n  {s['container']}, vol. {s['volume']}, "
              f"no. {s['issue']}, pp. {s['pages']}, {s['year']}")
        for k in ("title", "pages", "year"):
            mine = (row.get(k) or "").strip()
            if mine and norm(mine) != norm(s[k]):
                print(f"  ！ {k} が違う：メモ「{mine}」／Crossref「{s[k]}」")
                n_bad += 1
    print(f"\n{len(rows)} 件を確かめました。食い違い・取得できないもの：{n_bad} 件")


if __name__ == "__main__":
    main()

In [ ]:
%%capture cap
# ノートブック版：上のセルで定義した main() を、ターミナルで引数を変えて実行するのと同じように呼ぶ
import sys
for argv in [['crossref_check.py', '--csv', 'my_refs.csv']]:
    sys.argv = argv
    print('$ python', ' '.join(argv), flush=True)
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
if "取得できません" in cap.stdout and "Crossref「" not in cap.stdout:
    print("（Crossref につながらなかったので、確かめはとばします。インターネットにつながっているかを確かめてください）")
else:
    ok("pages が違う：メモ「1210-1214」／Crossref「1170-1174」" in cap.stdout,
       "窓長の論文で「pages が違う：メモ 1210-1214／Crossref 1170-1174」と表示される（my_refs.csv の例）")
    ok("Determined Blind Source Separation Unifying Independent Vector Analysis and Nonnegative Matrix Factorization" in cap.stdout
       and "1626-1641" in cap.stdout, "ILRMA の論文（TASLP 2016）の題とページ 1626-1641 が取れた")

## 3. 論文の図を 1 枚、小さく再現する（IV-9-3-4・3-5、手順 C、`repro_window.py`）

窓長の論文（Kitamura, Ono, Saruwatari, EUSIPCO 2017）の III-D 節の「STFT の窓長と分離性能」を、鏡像法の部屋・librosa の朗読 2 つ・2 マイクで小さく再現します。

まず小さく動かします（`--seeds 2 --win-ms 64 1024 --iter 5`、十数秒〜数十秒）。反復 5 回なので SDR は低くてよく、結果は `out/repro_try.csv` に書いて本番と分けます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
import sys
sys.argv = ['repro_window.py', '--seeds', '2', '--win-ms', '64', '1024', '--iter', '5', '--out', 'out/repro_try.csv']   # ノートブック版：ターミナルでの引数の代わり（Jupyter の引数を読まないように）
# 教科書：uv run python repro_window.py --seeds 2 --win-ms 64 1024 --iter 5 --out out/repro_try.csv
#!/usr/bin/env python3
"""IV-9：論文の図を1つ、小さく再現する（窓長と分離性能。Kitamura, Ono, Saruwatari, EUSIPCO 2017 の III-D 節）

使い方: cd ~/home-lab/partIV/iv9 && uv run python repro_window.py [--seeds 10] [--rt60 0.3] [--win-ms 32 64 128 256 512 1024]
再現したい主張（論文の III-A・III-D 節と結論。番号は本文の IV-9-3-4 の記入見本・7 節の表と同じ）
  (1) 窓が短すぎると、残響のために「周波数ごとの瞬時混合」の仮定が崩れ、どの方法も分離できない。
  (2) 窓が長すぎると、フレームの数 T が減り、推定（FDICA・IVA）が悪くなる（論文は 2T60 を超えると、と書く）。
  (4) 初期値が現実的（単位行列・乱数）なとき、ILRMA は長い窓で性能を保てず、最適な窓長は IVA とほぼ同じになる。
  （(3) 理想の初期値・音楽、(5) 基底の数、はこのプログラムでは扱わない）
この本での条件（論文との違いは本文の表で確かめる）
  pyroomacoustics の鏡像法（3次元の直方体）、16 kHz、T60 = 0.3 s（inverse_sabine に渡す設定値。論文の E2A に合わせた。
  measure_rt60 で測り直すと 0.33〜0.42 s）、マイク 2 本・間隔 5.66 cm、音源 2 つ・アレイの中心から 2 m・正面から ±50°
  （論文の図2(a) の配置をまねたもの）、音源は librosa の例の朗読 2 つ（libri1・libri3、8 秒）、ハミング窓、
  シフトは窓長の 1/4、反復 100 回、ILRMA の基底 K = 2。
  「種」は、部屋の大きさ・アレイの位置と ILRMA の初期値の乱数を、まとめて 1 つの数で決める（種＝部屋と初期値の組）。
  AuxIVA は単位行列から始めるので、種で変わるのは部屋だけ。音源分離の論文の「種 10 通り」（部屋を固定して初期値だけを
  変える）とは意味が違う。部屋と初期値を分けたいときは、simulate_fixed と ilrma に別々の種を渡すように直す。
  方法：AuxIVA（第7章）、ILRMA（第7章）、oracleW（IV-6 の bss6.oracle_w：正解の像 s を観測 x から最小二乗で回帰した
  分離行列 (Σ s xᴴ)(Σ x xᴴ)⁻¹。「周波数ごとの線形分離」の上限の目安で、推定の失敗を含まない。論文の式 (6) とは別の作り方）。
出力：out/repro_window.csv（1 行＝種×窓長×方法、2 音源の平均）、out/repro_window.png、
      out/repro_<方法>_<窓長>ms.wav（種 0。最大 −10 dBFS にそろえる）、画面に平均 ± 標準偏差、最適な窓長、誤用の例 2 つ
途中で止まったら --resume で続きから計算できる（CSV にある種と窓長はとばす）。
計算時間の目安：種 10 通り × 窓 6 通りで 40〜60 分（Apple シリコン の MacBook Air、執筆時）。まず --seeds 2 で動くことを確かめる。
"""
import argparse
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
from scipy import stats

HERE = pathlib.Path(".").resolve()   # ノートブック版：このノートブックのフォルダ（out/ もここ）
sys.path.insert(0, ".")   # ノートブック版：bss6.py・bss.py・common/ は、このノートブックの隣に書き出してある
import bss6                                            # noqa: E402
import common.plotting  # noqa: E402,F401              日本語フォント（Hiragino Sans）
from common.stft import stft, istft                    # noqa: E402
from common.audio_io import save                       # noqa: E402
from bss import auxiva, ilrma, demix                   # noqa: E402

OUT = HERE / "out"
OUT.mkdir(exist_ok=True)
FS = 16000
METHODS = ["AuxIVA", "ILRMA", "oracleW"]


def simulate_fixed(src, rt60, seed, d=0.0566, dist=2.0, deg=50.0):
    """論文の図2(a) に似た配置：2 本のマイク（間隔 d）の正面から ±deg 度、距離 dist の 2 音源。
    部屋（幅 5〜8 m・奥行 4〜6 m・高さ 3 m）とアレイの位置は種で変える。戻り値は bss6.simulate と同じ (mix, img)"""
    import pyroomacoustics as pra
    rng = np.random.default_rng(seed)
    dim = np.array([rng.uniform(5, 8), rng.uniform(4, 6), 3.0])
    e_abs, max_order = pra.inverse_sabine(rt60, dim)
    room = pra.ShoeBox(dim, fs=FS, materials=pra.Material(e_abs), max_order=max_order)
    c = np.array([dim[0] / 2 + rng.uniform(-0.5, 0.5), dim[1] / 2 + rng.uniform(-0.5, 0.5), 1.5])
    room.add_microphone_array(np.stack([c + [-d / 2, 0, 0], c + [d / 2, 0, 0]], axis=1))
    for k, a in enumerate(np.radians([90 - deg, 90 + deg])):       # x 軸がアレイの向き、90° が正面
        room.add_source(c + dist * np.array([np.cos(a), np.sin(a), 0.0]), signal=src[k])
    pre = room.simulate(return_premix=True)                          # (音源, マイク, 時間)
    n = src.shape[1]
    return pre.sum(axis=0)[:, :n], pre[:, 0, :n]


def separate(mix, img, L, seed, n_iter, K):
    """窓長 L（シフト L/4、ハミング窓）で STFT し、3 つの方法で分離して時間信号に戻す"""
    S = L // 4
    n = mix.shape[1]
    X = stft(mix.T, L, S, window="hamming")                          # (F, T, M)
    Simg = stft(img.T, L, S, window="hamming")                       # (F, T, N)
    out = {}
    t0 = time.time()
    out["AuxIVA"] = (auxiva(X, n_iter)[0], time.time() - t0)
    t0 = time.time()
    out["ILRMA"] = (ilrma(X, n_basis=K, n_iter=n_iter, seed=seed)[0], time.time() - t0)
    t0 = time.time()
    out["oracleW"] = (demix(bss6.oracle_w(X, Simg), X), time.time() - t0)
    res = {}
    for m, (Y, sec) in out.items():
        y = np.stack([istft(Y[:, :, k], L, S, window="hamming", length=n) for k in range(Y.shape[2])])
        res[m] = (y, sec, X.shape[1])
    return res


def summarize(df):
    wins = sorted(df.win_ms.unique())
    tab = df.groupby(["method", "win_ms"])["sdr_imp"].agg(["mean", "std"])
    print("\nSDR 改善量 [dB]：平均 ± 標準偏差（種＝部屋と初期値の組 %d 通り）" % df.seed.nunique())
    print("  窓長[ms] " + "".join(f"{w:>14.0f}" for w in wins))
    print("  フレーム数 " + "".join(f"{int(df.loc[df.win_ms == w, "T"].iloc[0]):>12d}  " for w in wins))
    for m in METHODS:
        print(f"  {m:8s} " + "".join(f"{tab.loc[(m, w), 'mean']:7.2f}±{tab.loc[(m, w), 'std']:4.2f} " for w in wins))
    # (4) の主張：種ごとの最適な窓長を、AuxIVA と ILRMA で比べる（同じ種どうしの対応のある比較）
    best = df.loc[df.groupby(["method", "seed"])["sdr_imp"].idxmax(), ["method", "seed", "win_ms"]]
    bw = best.pivot(index="seed", columns="method", values="win_ms")
    print("\n種ごとの最適な窓長 [ms]（中央値）：" + "、".join(f"{m} {bw[m].median():.0f}" for m in METHODS))
    ratio = np.log2(bw["ILRMA"] / bw["AuxIVA"])
    print(f"  log2(ILRMA の最適 / AuxIVA の最適) の分布：{dict(ratio.value_counts().sort_index())}"
          f"（0 なら同じ窓長。1 は 2 倍）")
    # 平均の曲線で見た最適（論文の図と同じ見方）
    for m in METHODS:
        mu = tab.loc[m, "mean"]
        print(f"  平均の曲線で最大になる窓長：{m} {mu.idxmax():.0f} ms（{mu.max():.2f} dB）")
    # ILRMA − AuxIVA（窓長ごと、同じ種の差）
    print("\nILRMA − AuxIVA（同じ種の差）：平均と 95% t 区間")
    for w in wins:
        p = df[df.win_ms == w].pivot(index="seed", columns="method", values="sdr_imp")
        d = (p["ILRMA"] - p["AuxIVA"]).to_numpy()
        ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(), scale=d.std(ddof=1) / np.sqrt(len(d)))
        print(f"  {w:5.0f} ms：{d.mean():+.2f} dB [{ci[0]:+.2f}, {ci[1]:+.2f}]、ILRMA が上の種 {np.sum(d > 0)}/{len(d)}")
    misuse(df)
    return tab, wins


def misuse(df, n_split=200, seed=0):
    """統計の誤用の例を2つ計算する。
    (1) 平均 ± 標準偏差の重なりで「差がない」と言う（対応のない比較）。同じ種どうしの差（対応のある比較）と並べる。
    (2) 評価に使う同じ種で窓長を選び、その成績を報告する（選択のかたより）。種を半分に分け、前半で選んだ窓長の
        前半での成績（誤り）と後半での成績（正しい）の差を、分け方を変え入れ替えも含めて平均する。"""
    print("\n統計の誤用（1）：平均 ± SD の重なりで比べる（対応なし）か、同じ種の差で比べる（対応あり）か")
    for w in sorted(df.win_ms.unique()):
        p = df[df.win_ms == w].pivot(index="seed", columns="method", values="sdr_imp")
        a, b = p["ILRMA"].to_numpy(), p["AuxIVA"].to_numpy()
        welch = stats.ttest_ind(a, b, equal_var=False).pvalue
        paired = stats.ttest_rel(a, b).pvalue
        print(f"  {w:5.0f} ms：ILRMA {a.mean():5.2f}±{a.std(ddof=1):4.2f}、AuxIVA {b.mean():5.2f}±{b.std(ddof=1):4.2f}"
              f" → 対応なし（Welch）p = {welch:.4f}、対応あり p = {paired:.4f}、相関 {np.corrcoef(a, b)[0, 1]:+.2f}")
    rng = np.random.default_rng(seed)
    seeds = np.array(sorted(df.seed.unique()))
    if len(seeds) < 4:
        print("\n（誤用（2）は種 4 つ以上で計算します）")
        return
    print("\n統計の誤用（2）：評価に使う種で窓長を選ぶ（分け方 %d 通り×入れ替え）" % n_split)
    for m in ["AuxIVA", "ILRMA"]:
        p = df[df.method == m].pivot(index="seed", columns="win_ms", values="sdr_imp")
        bias, chosen = [], []
        for _ in range(n_split):
            perm = rng.permutation(seeds)
            a, b = perm[:len(seeds) // 2], perm[len(seeds) // 2:]
            for x, y in ((a, b), (b, a)):
                w = p.loc[x].mean().idxmax()
                chosen.append(w)
                bias.append(p.loc[x, w].mean() - p.loc[y, w].mean())
        share = pd.Series(chosen).value_counts(normalize=True)
        print(f"  {m:7s}：かたよりの平均 {np.mean(bias):+.3f} dB（標準偏差 {np.std(bias):.2f}）。選ばれた窓長の割合 "
              + "、".join(f"{k:.0f} ms {v:.0%}" for k, v in share.items()))
        if len(share) == 1:
            print("           （どの分け方でも同じ窓長が選ばれたので、前半と後半を入れ替えて平均するこの比べ方では、"
                  "かたよりは作り方から必ず 0 になる）")
        print(f"           種ごとに正解を見て一番よい窓長を選んだ平均 {p.max(axis=1).mean():.2f} dB"
              f"（固定した最良の窓長 {p.mean().max():.2f} dB）")


def plot(df, tab, wins):
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    style = {"AuxIVA": ("o-", "C0"), "ILRMA": ("s-", "C3"), "oracleW": ("^--", "0.4")}
    for m in METHODS:
        mu = np.array([tab.loc[(m, w), "mean"] for w in wins])
        sd = np.array([tab.loc[(m, w), "std"] for w in wins])
        mk, col = style[m]
        ax.plot(wins, mu, mk, color=col, label=m)
        ax.fill_between(wins, mu - sd, mu + sd, color=col, alpha=0.12)
    ax.set_xscale("log", base=2)
    ax.set_xticks(wins)
    ax.set_xticklabels([f"{w:.0f}" for w in wins])
    rt = df.rt60.iloc[0]
    ax.axvline(2 * rt * 1000, color="k", lw=0.8, ls=":")
    ax.text(2 * rt * 1000, ax.get_ylim()[1], " 2×T60（設定値）", va="top", fontsize=9)
    ax.set_xlabel("STFT の窓長 [ms]（シフトは 1/4）")
    ax.set_ylabel("SDR 改善量 [dB]（平均 ± 標準偏差）")
    ax.set_title(f"窓長と分離性能（T60 の設定値 {rt * 1000:.0f} ms、種＝部屋と初期値の組 {df.seed.nunique()} 通り）")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "repro_window.png", dpi=120)
    plt.show()   # ノートブック版：図を表示する
    print("out/repro_window.png を保存しました")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--rt60", type=float, default=0.3)
    ap.add_argument("--win-ms", type=float, nargs="+", default=[32, 64, 128, 256, 512, 1024])
    ap.add_argument("--iter", type=int, default=100)
    ap.add_argument("--K", type=int, default=2)
    ap.add_argument("--sec", type=float, default=8.0)
    ap.add_argument("--out", default=str(OUT / "repro_window.csv"))
    ap.add_argument("--summary-only", action="store_true", help="計算せず、保存した CSV をまとめ直すだけ")
    ap.add_argument("--resume", action="store_true", help="途中で止まったとき、CSV にある（種, 窓長）はとばして続ける")
    a = ap.parse_args()
    out_csv = pathlib.Path(a.out)
    if a.summary_only:
        df = pd.read_csv(out_csv)
        plot(df, *summarize(df))
        return
    src = bss6.sources(sec=a.sec)                                   # libri1・libri3、標準偏差 1 にそろえる
    rows = pd.read_csv(out_csv).to_dict("records") if a.resume and out_csv.exists() else []
    done = {(r["seed"], r["win_ms"]) for r in rows}
    for seed in range(a.seeds):
        mix, img = simulate_fixed(src, a.rt60, seed)
        sdr_in = float(np.mean(bss6.evaluate(img, np.stack([mix[0], mix[0]]))[0]))
        for wm in a.win_ms:
            if (seed, wm) in done:
                continue
            L = int(round(wm / 1000 * FS))
            t0 = time.time()
            res = separate(mix, img, L, seed, a.iter, a.K)
            for m in METHODS:
                y, sec, T = res[m]
                sdr, sir, sar = bss6.evaluate(img, y)
                rows.append(dict(rt60=a.rt60, seed=seed, win_ms=wm, L=L, T=T, method=m, sdr_in=sdr_in,
                                 sdr=float(np.mean(sdr)), sir=float(np.mean(sir)), sar=float(np.mean(sar)),
                                 sdr_imp=float(np.mean(sdr)) - sdr_in, sec=sec))
                if seed == 0 and m != "oracleW":
                    save(OUT / f"repro_{m}_{wm:.0f}ms.wav", y[0] / np.max(np.abs(y[0])) * 10 ** (-10 / 20), FS)
            msg = "  ".join(f"{r['method']} {r['sdr_imp']:+.1f}" for r in rows[-3:])
            print(f"種 {seed} 窓 {wm:5.0f} ms（T = {rows[-1]['T']}、{time.time() - t0:4.0f} 秒）SDR 改善量 [dB]：{msg}",
                  flush=True)
            pd.DataFrame(rows).to_csv(out_csv, index=False)
    df = pd.DataFrame(rows)
    plot(df, *summarize(df))


if __name__ == "__main__":
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
import pandas as pd, os
d = pd.read_csv("out/repro_try.csv")
ok(len(d) == 2 * 2 * 3, f"out/repro_try.csv は 種 2 × 窓長 2 × 方法 3 ＝ {len(d)} 行（画面の「種 … 窓 …」の行は 4 行）")
ok(len(re.findall(r"^種 \d 窓", cap.stdout, re.M)) == 4 and "平均 ± 標準偏差" in cap.stdout and "out/repro_window.png を保存しました" in cap.stdout,
   "4 行の結果と、平均 ± 標準偏差の表、out/repro_window.png の保存の表示が出た（反復 5 回なので SDR は低くてよい）")
o = d[d.method == "oracleW"].groupby("win_ms")["sdr_imp"].mean()
ok(o[1024] > o[64] + 10, f"oracleW（推定を使わない上限の目安）は反復の回数に関係なく、1024 ms のほうがずっと高い（64 ms {o[64]:.1f}・1024 ms {o[1024]:.1f} dB）")

次が「本番」にあたるセルです。**ノートブックの既定は試しの設定 `--seeds 4`**（種 4 通り × 窓長 32〜1024 ms の 6 通り、反復 100 回。執筆時の Mac で約 4 分）です。教科書の本番（種 10 通り、40〜60 分）にするときは、`--seeds", "4"` を消して `["repro_window.py"]` にしてください。途中で止まったら、`["repro_window.py", "--resume"]` で続きから計算できます。

結果の `out/repro_window.csv` は 4 節（手順 D）で使います。7 節の表（種 10 通り）と見比べてください。

In [ ]:
%%capture cap
# ノートブック版：上のセルで定義した main() を、ターミナルで引数を変えて実行するのと同じように呼ぶ
# 本番（教科書の手順 C の 2。種 10 通り、40〜60 分）：for argv in [["repro_window.py"]]:
# 途中で止まったら：for argv in [["repro_window.py", "--resume"]]:
import sys
for argv in [['repro_window.py', '--seeds', '4']]:
    sys.argv = argv
    print('$ python', ' '.join(argv), flush=True)
    main()

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
import pandas as pd
d = pd.read_csv("out/repro_window.csv")
ns = d.seed.nunique()
ok(len(d) == ns * 6 * 3, f"out/repro_window.csv は 種 {ns} × 窓長 6 × 方法 3 ＝ {len(d)} 行（教科書の本番は種 10 で 180 行）")
T = d.groupby("win_ms")["T"].first().astype(int).tolist()
ok(T == [1001, 501, 251, 126, 63, 32], f"フレームの数 T が 1001・501・251・126・63・32（{T}）")
m = d.groupby(["method", "win_ms"])["sdr_imp"].mean().unstack()
ok(m.loc["oracleW"].is_monotonic_increasing, f"oracleW は窓が長いほど上がり続ける（{m.loc['oracleW'].round(2).tolist()}。7 節 8.88〜27.03）")
ok(m.loc["AuxIVA"].idxmax() in (64, 128) and m.loc["ILRMA"].idxmax() in (256, 512),
   f"平均の曲線で最大になる窓長：AuxIVA {m.loc['AuxIVA'].idxmax():.0f} ms・ILRMA {m.loc['ILRMA'].idxmax():.0f} ms（7 節：AuxIVA 128 ms、ILRMA 256 ms）")
ok(m.loc["AuxIVA", 1024] < 0 and m.loc["ILRMA", 1024] < 1,
   f"1024 ms（T = 32）では AuxIVA の改善量が負、ILRMA もほぼ 0 以下（{m.loc['AuxIVA', 1024]:.2f}・{m.loc['ILRMA', 1024]:.2f} dB。7 節 −1.65・−0.40）")
p = d[d.win_ms == 256].pivot(index="seed", columns="method", values="sdr_imp")
ok(bool((p["ILRMA"] > p["AuxIVA"]).all()), f"256 ms では、どの種でも ILRMA が AuxIVA より上（差の平均 {(p['ILRMA'] - p['AuxIVA']).mean():+.2f} dB。7 節 +8.39）")
ref = {("AuxIVA", 128): 5.62, ("ILRMA", 128): 8.67, ("oracleW", 128): 12.47, ("ILRMA", 256): 11.78, ("oracleW", 256): 16.19}
dev = {k: round(m.loc[k] - v, 2) for k, v in ref.items()}
ok(all(abs(v) < 2.0 for v in dev.values()), f"128・256 ms の平均が 7 節の表（種 10 通り）から 2 dB 以内（ずれ {dev}）")
ok("out/repro_window.png を保存しました" in cap.stdout, "out/repro_window.png ができた")

**論文の主張と比べる**（手順 C の 3）：7 節の表「論文の主張と再現の結果」の形で、主張 (1)・(2)・(4) のそれぞれに「再現できた・一部・できなかった」と、条件の違い（実測とシミュレーション、音声の種類、乱数の初期値の作り方）をノートに書いてください。試しの設定（種 4 通り）では、種ごとの最適な窓長の数え方が 7 節（10 通り）と違います。

**耳で確かめる**（手順 C の 4、音量は小さく）：種 0 の ILRMA の 64・256・1024 ms と、同じ窓長の AuxIVA を聞き比べます。256 ms の ILRMA でもう一方の声が最も小さいこと、1024 ms ではどちらの方法でも 2 人の声がほとんど分かれていないことが聞き取れるでしょうか。wav は 1 つずつ最大値を −10 dBFS にそろえているので、ファイルどうしの音量の差には意味がありません。

In [ ]:
# 保存した wav を聞く（音量を小さくしてから。1つずつ）
for f in ['out/repro_ILRMA_64ms.wav', 'out/repro_ILRMA_256ms.wav', 'out/repro_ILRMA_1024ms.wav', 'out/repro_AuxIVA_64ms.wav', 'out/repro_AuxIVA_256ms.wav', 'out/repro_AuxIVA_1024ms.wav']:
    print(f)
    listen_file(f)

## 4. 必要な試行の数を出す（IV-9-3-6、手順 D の 2、`seeds_needed.py`）

3 つ実行します。(a) 3 節の CSV を予備実験として、窓長 128 ms の ILRMA − AuxIVA の差の標準偏差 $\sigma_d$ から $\delta = 0.5$ dB に要る試行の数を出す（教科書の手順 D のコマンド。予備実験が種 4 通りなので、$\sigma_d$ の区間は 7 節の 10 通りのときより広くなります）。(b) 7 節の $\sigma_d = 0.74$ dB を直接与える。(c) 既定の $\sigma_d = 1$ dB（7 節の「試行の数とワークシート」の表）。上で書き出した `seeds_needed.py` を `%run` で実行します。

In [ ]:
%%capture cap
# ノートブック版：書き出したファイルを、教科書のコマンドと同じ引数で実行する
print('$ python seeds_needed.py --pilot out/repro_window.csv --a ILRMA --b AuxIVA --where win_ms=128 --delta 0.5', flush=True)
%run seeds_needed.py --pilot out/repro_window.csv --a ILRMA --b AuxIVA --where win_ms=128 --delta 0.5
print('$ python seeds_needed.py --sd-diff 0.74 --delta 0.5', flush=True)
%run seeds_needed.py --sd-diff 0.74 --delta 0.5
print('$ python seeds_needed.py', flush=True)
%run seeds_needed.py

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
parts = cap.stdout.split("$ python seeds_needed.py")
ok(len(parts) == 4, "seeds_needed.py を 3 回実行した")
a, b, c = parts[1], parts[2], parts[3]
ok("試行 " in a and re.search(r"σ_d の 95% 区間（正規分布を仮定）\[[\d.]+, [\d.]+\] dB → 要る試行の数は \d+〜\d+", a) is not None,
   "(a) 3 節の CSV から σ_d と、その 95% 区間・要る試行の数の幅が表示された（試しの設定なので 7 節の [0.51, 1.35]・11〜59 とは違う）")
sd_a = num(r"σ_d = ([\d.]+) dB\n", text=a)
ok(0.2 < sd_a < 2.0, f"(a) σ_d = {sd_a} dB（7 節は種 10 通りで 0.74 dB）")
nb_ = re.search(r"正規近似 ([\d.]+) → (\d+)、非心 t（自前）(\d+)、statsmodels ([\d.]+) → (\d+)", b).groups()
ok(nb_[2] == "20" and nb_[4] == "20", f"(b) σ_d = 0.74 dB で、非心 t で 20、statsmodels で {nb_[3]} → 20（7 節は予備実験の丸める前の σ_d で 19.08 → 20、正規近似 17.07 → 18。ここは 0.74 ちょうどなので {nb_[0]} → {nb_[1]}）")
nc = re.search(r"正規近似 ([\d.]+) → (\d+)、非心 t（自前）(\d+)、statsmodels ([\d.]+) → (\d+)", c).groups()
ok(nc == ("31.40", "32", "34", "33.37", "34"), f"(c) σ_d = 1 dB で 正規近似 31.40 → 32、非心 t 34、statsmodels 33.37 → 34（{nc}）")
ok("自前の n = 34 での検出力 0.8078、statsmodels の検出力 0.8078" in c, "(c) n = 34 での検出力が自前・statsmodels とも 0.8078")
mde = [float(x) for x in re.findall(r"正確な t ([\d.]+) dB（statsmodels", c)]
ok(mde == [1.682, 0.996, 0.660, 0.454], f"(c) 見つけられる最小の差が n = 5・10・20・40 で 1.682・0.996・0.660・0.454 dB（{mde}）")

**事前登録**（手順 D の 3）：教科書では、ここで最終演習K の K-4-5 のひな形に、仮説・主な解析・試行の数とその根拠・判定の基準を書き、実験を回す前に Git にコミットします（`git log` で、事前登録のコミットが結果の CSV のコミットより前にあることを確かめる）。ノートブックでは行いません。

## 5. テーマ候補のワークシート（IV-9-3-7〜3-9、手順 E の 3、`theme_sheet.py`）

見本の候補 3 つで、合計点と判定（有力・候補・近すぎ・遠すぎ）、記入の抜けの点検を表示し、`out/theme_sheet.md` を作ります。`--blank` は空欄のひな形 `out/theme_sheet_blank.md` です。自分の候補にするときは、上の `%%writefile theme_sheet.py` のセルの `CANDIDATES` を書きかえ、そのセルとこのセルを実行し直してください（見本の文をそのまま残さない）。

In [ ]:
%%capture cap
# ノートブック版：書き出したファイルを、教科書のコマンドと同じ引数で実行する
print('$ python theme_sheet.py', flush=True)
%run theme_sheet.py
print('$ python theme_sheet.py --blank', flush=True)
%run theme_sheet.py --blank

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
v = re.findall(r"合計 (\d+)/12 → (\S+)", cap.stdout)
ok(v == [("11", "有力"), ("10", "有力"), ("10", "有力")], f"見本の 3 つが ① 11/12・② 10/12・③ 10/12、どれも「有力」（{v}）")
ok("点検：" not in cap.stdout, "「点検：」の行が 1 つも出ない")
import os
ok(os.path.exists("out/theme_sheet.md") and os.path.exists("out/theme_sheet_blank.md"), "out/theme_sheet.md と out/theme_sheet_blank.md ができた")

In [ ]:
# できたワークシートの最初の 30 行を見る（研究室訪問の前日に、印刷するか PDF にして持っていく）
print("\n".join(open("out/theme_sheet.md", encoding="utf-8").read().splitlines()[:30]))

## 6. 確認問題の答え合わせ（IV-9-7、`quiz_check_iv9.py`）

**先に紙で解いてから**実行してください。問1〜問7 の数値の答えを計算します（問3 は手順 B の段階3 の確かめも兼ねます）。上で書き出した `seeds_needed.py` と `theme_sheet.py` を読み込みます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
#!/usr/bin/env python3
"""IV-9：確認問題の数値の答えを計算して確かめる（7 節の解答と同じ値が出る）

使い方: cd ~/home-lab/partIV/iv9 && uv run python quiz_check_iv9.py
"""
import numpy as np
from scipy import stats

import seeds_needed as sn
from theme_sheet import judge

print("問1 差の標準偏差と、要る試行の数")
sd = sn.sd_diff(2.5, 2.0, 0.8)
nn, x = sn.n_normal(0.5, sd)
print(f"  σ_d = {sd:.3f} dB、正規近似 n = {x:.1f} → {nn}、非心 t で {sn.n_exact(0.5, sd)}")
nn0, x0 = sn.n_normal(0.5, sn.sd_diff(2.5, 2.0, 0.0))
print(f"  対応をとらない（ρ = 0 とみなす）と σ = {sn.sd_diff(2.5, 2.0, 0.0):.3f} dB、n ≈ {x0:.1f} → {nn0}")

print("\n問2 平均 ± SD だけでは差が言えない。差の標準偏差が 0.6 dB なら")
d_mean, d_sd, n = 0.6, 0.6, 10
t = d_mean / (d_sd / np.sqrt(n))
p = 2 * stats.t.sf(abs(t), n - 1)
ci = stats.t.interval(0.95, n - 1, loc=d_mean, scale=d_sd / np.sqrt(n))
print(f"  t = {t:.3f}、自由度 {n - 1}、両側 p = {p:.4f}、95% 区間 [{ci[0]:.3f}, {ci[1]:.3f}] dB")
# 同じ平均と SD を、対応のない 2 群（Welch）と誤って扱った場合
w = stats.ttest_ind_from_stats(8.2, 2.1, 10, 7.6, 2.3, 10, equal_var=False)
print(f"  （誤り）対応のない Welch の t 検定：t = {w.statistic:.3f}、p = {w.pvalue:.3f}")

print("\n問3 oracle の分離行列 W = (Σ s xᴴ)(Σ x xᴴ)⁻¹ の確かめ（雑音なしなら A⁻¹ に一致）")
rng = np.random.default_rng(1)
M, T = 2, 200
A = rng.normal(size=(M, M)) + 1j * rng.normal(size=(M, M))
s = rng.laplace(size=(M, T)) + 1j * rng.laplace(size=(M, T))
x = A @ s
W = (s @ x.conj().T) @ np.linalg.inv(x @ x.conj().T)
print(f"  |W − A⁻¹| の最大 = {np.max(np.abs(W - np.linalg.inv(A))):.2e}")
xn = x + 0.3 * (rng.normal(size=x.shape) + 1j * rng.normal(size=x.shape))
Wn = (s @ xn.conj().T) @ np.linalg.inv(xn @ xn.conj().T)
Wl = np.linalg.lstsq(xn.T, s.T, rcond=None)[0].T                   # 最小二乗の解（行ごと）と同じか
print(f"  雑音ありでも最小二乗の解と一致：差の最大 = {np.max(np.abs(Wn - Wl)):.2e}（A⁻¹ とは {np.max(np.abs(Wn - np.linalg.inv(A))):.3f} ずれる）")

print("  論文の式 (6)：W6 = (Σ s sᴴ)(Σ x sᴴ)⁻¹（x を s で回帰した混合行列の推定 Â の逆行列）と比べる")
W6 = (s @ s.conj().T) @ np.linalg.inv(x @ s.conj().T)
print(f"  雑音なし：|W6 − A⁻¹| の最大 = {np.max(np.abs(W6 - np.linalg.inv(A))):.2e}（どちらも A⁻¹）")
W6n = (s @ s.conj().T) @ np.linalg.inv(xn @ s.conj().T)
err = lambda W: float(np.sum(np.abs(s - W @ xn) ** 2))                    # noqa: E731
print(f"  雑音あり：|W6 − W★| の最大 = {np.max(np.abs(W6n - Wn)):.3f}、2 乗誤差 Σ‖s − W x‖²：W★ {err(Wn):.1f}、W6 {err(W6n):.1f}"
      f"（W★ は 2 乗誤差が最小になるように作ったので、必ず W6 以下）")

print("\n問4 窓長とフレームの数（16 kHz、8 秒、シフトは窓長の 1/4）")
fs, nsamp = 16000, 8 * 16000
for ms in (128, 512, 600, 1024):
    L = int(ms / 1000 * fs)
    S = L // 4
    print(f"  {ms:5d} ms：L = {L}、S = {S}、T = 1 + n//S = {1 + nsamp // S}")

print("\n問5 題目の割合（分離の数理：1〜3年目 6/7、4〜6年目 2/8。見本の題目）")
tab = [[6, 1], [2, 6]]
odds, p = stats.fisher_exact(tab)
print(f"  フィッシャーの正確検定 p = {p:.4f}（オッズ比 {odds:.2f}）。7 つのタグで同じことをすると、"
      f"少なくとも 1 つが偶然 p < 0.05 になる確率は独立なら 1 − 0.95⁷ = {1 - 0.95 ** 7:.3f}")

print("\n問6 多重比較：差の無い 36 条件をそれぞれ α = 0.05 で検定")
k = 36
print(f"  偽陽性の期待値 {k * 0.05:.2f}、少なくとも 1 つ（独立とみなす）{1 - 0.95 ** k:.3f}、"
      f"ボンフェローニの水準 {0.05 / k:.5f}")

print("\n問7 テーマの距離の判定（theme_sheet.judge）")
cases = {
    "見本の卒研（発話区間の検出）を同じ条件でやり直す": dict(Q=2, LAB=2, NEW=0, M1=2, ME=1, ALT=1),
    "株価の予測に深層学習を使う": dict(Q=2, LAB=0, NEW=2, M1=2, ME=2, ALT=2),
    "話す時間の割合と BSS の関係を、種と部屋の混合モデルで区間推定する": dict(Q=2, LAB=2, NEW=1, M1=2, ME=2, ALT=1),
}
for name, sc in cases.items():
    print(f"  {name}：合計 {judge(sc)[0]} → {judge(sc)[1]}")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す
import re
# 確かめ（教科書の手順の「確かめること」と 7 節の結果の例）
ok("σ_d = 1.500 dB、正規近似 n = 70.6 → 71、非心 t で 73" in cap.stdout and "n ≈ 321.8 → 322" in cap.stdout,
   "問1：σ_d = 1.5 dB、70.6 → 71（非心 t で 73）、対応をとらないと 321.8 → 322")
ok("t = 3.162、自由度 9、両側 p = 0.0115、95% 区間 [0.171, 1.029] dB" in cap.stdout and "t = 0.609、p = 0.550" in cap.stdout,
   "問2：t = 3.162、p = 0.0115、[0.171, 1.029] dB、Welch で p = 0.550")
e1 = num(r"\|W − A⁻¹\| の最大 = ([\d.e+-]+)")
e2 = num(r"\|W6 − A⁻¹\| の最大 = ([\d.e+-]+)")
ok(e1 < 1e-14 and e2 < 1e-14, f"問3：雑音なしで |W − A⁻¹|・|W6 − A⁻¹| の最大が 10⁻¹⁵ 程度（{e1:.1e}・{e2:.1e}。7 節 8.2e-16・3.7e-16）")
ok("W★ 58.4、W6 62.1" in cap.stdout, "問3：雑音ありの 2 乗誤差 W★ 58.4、W6 62.1")
T = [int(x) for x in re.findall(r"T = 1 \+ n//S = (\d+)", cap.stdout)]
ok(T == [251, 63, 54, 32], f"問4：T が 128 ms 251・512 ms 63・600 ms 54・1024 ms 32（{T}）")
ok("p = 0.0406（オッズ比 18.00）" in cap.stdout and "= 0.302" in cap.stdout, "問5：p = 0.0406（オッズ比 18.00）、1 − 0.95⁷ = 0.302")
ok("偽陽性の期待値 1.80、少なくとも 1 つ（独立とみなす）0.842、ボンフェローニの水準 0.00139" in cap.stdout, "問6：1.80・0.842・0.00139")
j = re.findall(r"合計 (\d+) → (近すぎ|遠すぎ|有力|候補)", cap.stdout)
ok(j == [("8", "近すぎ"), ("10", "遠すぎ"), ("10", "有力")], f"問7：(a) 8 近すぎ、(b) 10 遠すぎ、(c) 10 有力（{j}）")

## おわりに

- 研究地図の数え方と分類の規則の影響、書誌の確かめ、論文の図の小さな再現（oracle の上限と推定の差、同じ種どうしの比較、統計の誤用）、必要な試行の数、テーマ候補の点検を確かめました。
- 本当の再現の結論（7 節の表と比べられるもの）は、3 節を本番の設定（種 10 通り）にすると得られます。改良の実験は、**事前登録（コミット）をしてから**回してください。
- このノートブックに入れていないもの：研究室のページ・KAKEN・業績一覧を読むこと、論文を 3 段階で読むこと、事前登録とコミット、問いの種の書き出しと採点、研究室訪問の準備（手順 F）。